# LifeMate — lm-sft-038: smoke test Qwen3.5-4B (train LoRA trên T4 → merge → vLLM)

**Import → Save & Run All.** Dataset LifeMate, wheels (`build-wheels` v4), GPU T4 ×2 và Internet đã gắn sẵn trong `.ipynb`. **Kiểm Settings → Internet: On** trước khi chạy. Không cần Add Input gì thêm. Thời gian dự kiến khoảng 1.5–2.5 giờ.

Câu hỏi cần trả lời trước khi bỏ ~20 giờ GPU để train Qwen3.5-4B đủ 2 epoch (so với Qwen3-4B-Instruct-2507 của run 021/023):
1. Qwen3.5-4B (24 lớp linear attention Gated DeltaNet + 8 lớp attention, vocab 248k) **có train LoRA được trên T4 fp16 không**: hết VRAM? loss có NaN không?
2. **Kernel nào chạy được**: Triton `fla` (nhanh, chưa chắc chạy trên sm75) hay đường torch của transformers (chậm hơn). Kiểm số học: loss hai đường phải gần nhau.
3. **Tốn bao nhiêu giây/step** so với Qwen3-4B trên cùng 192 example, cùng công thức 021 (LoRA r16/α32, lr 1e-4, accum 16, max_len 8192) → ước tính số giờ cho 2 epoch.
4. **Merge + vLLM**: adapter merge vào checkpoint gốc rồi vLLM 0.32 (eager, `language_model_only` như run 011) có load và sinh JSON hợp lệ không.

Bố trí: **GPU0** = Qwen3.5-4B (probe đường torch → probe đường fla → train 12 step bằng đường tốt hơn → merge CPU → vLLM), **GPU1** = Qwen3-4B-Instruct-2507 (probe + train 12 step, làm mốc tốc độ/VRAM). Dữ liệu: train_fit (seed 2026), transcript gold, aug như 021. 12 step chỉ đo tốc độ/độ ổn định, **không đo chất lượng**; chất lượng đo ở bước so sánh 2 epoch.

Kết quả: bảng cuối notebook + `lm_q35smoke/SMOKE.json` (`verdict` GO / NO-GO kèm lý do). Gửi lại bảng cuối (hoặc file SMOKE.json) để ghi vào tracking và quyết định bước tiếp.


In [ ]:
CONFIG = {
  "track": "lifemate",
  "experiment": "lm-sft-038-qwen35-smoke",
  "split": "train",
  "manifest_seed": 2026,
  "input_roots": ["/kaggle/input"],
  "out_dir": "/kaggle/working/lm_q35smoke",
  # LLM < 5B: Qwen3.5-4B 4,659,865,088 tham số (gồm vision), Apache-2.0, tạo 2026-02-27 (HF API, kiểm 2026-11-02)
  "candidate": {"model": "Qwen/Qwen3.5-4B", "revision": "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"},
  "baseline": {"model": "Qwen/Qwen3-4B-Instruct-2507", "revision": "cdbee75f17c01a7cc42f958dc650907174af0554"},
  "result_chars": 1500,
  # công thức lm-sft-021 ep2 (dev 36.47); max_steps chỉ cho smoke
  "sft": {"lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.05, "lr": 1e-4, "warmup_ratio": 0.03, "grad_accum": 16,
          "max_len": 8192, "gradient_checkpointing": True, "seed": 0, "ckpt_every_steps": 1000, "ckpt_every_min": 1000,
          "wallclock_s": 3 * 3600},
  "smoke_steps": 12,
  "aug": {"aug": True, "drop_p": 0.3},
  # Qwen3.5: q/k/v/o ở 8 lớp full attention, in_proj_qkv/in_proj_z/out_proj ở 24 lớp Gated DeltaNet, MLP mọi lớp
  "target_modules": {
    "candidate": ["q_proj", "k_proj", "v_proj", "o_proj", "in_proj_qkv", "in_proj_z", "out_proj", "gate_proj", "up_proj", "down_proj"],
    "baseline": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
  },
  "fla_max_loss_diff": 0.05,  # |loss fla − loss torch| tối đa trên 3 example probe để nhận kernel fla
  "vllm_n": 48,
  # như run 011 (Qwen3.5 trên T4): eager, chỉ phần ngôn ngữ, prefill chunk 1024
  "vllm": {"dtype": "half", "max_model_len": 16384, "gpu_memory_utilization": 0.85, "enable_thinking": False, "seed": 0,
           "enforce_eager": True, "language_model_only": True, "max_num_seqs": 4, "max_num_batched_tokens": 1024,
           "enable_chunked_prefill": True},
  "vllm_spec": "vllm==0.32.0 wrapt",
}


In [ ]:
import os, sys, json, time, glob, shutil, subprocess, threading
from pathlib import Path
WORK = Path("/kaggle/working")
LIB = WORK / "lib"
(LIB / "lmlib").mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LIB))
OUT = Path(CONFIG["out_dir"]); OUT.mkdir(parents=True, exist_ok=True)
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"  # zaic_tools lives on the read-only input

def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True, capture_output=True, text=True); print((r.stdout + r.stderr)[-3000:]); return r.returncode


## Thư viện `lmlib` (nhúng sẵn, giống run 021 + `qwen35smoke`)

In [ ]:
%%writefile /kaggle/working/lib/lmlib/__init__.py
"""LifeMate library embedded into the Kaggle notebooks (see build_notebooks.py)."""

In [ ]:
%%writefile /kaggle/working/lib/lmlib/common.py
"""Run plumbing shared by the LifeMate notebooks: durable JSONL shards with resume, stable sharding, config hashes,
atomic writes, input discovery under /kaggle/input and environment info."""
from __future__ import annotations

import hashlib
import json
import os
import platform
import subprocess
import time
from pathlib import Path


def dumps(obj) -> str:
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"), allow_nan=False, default=str)


def load_jsonl(path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def read_jsonl_tolerant(path) -> tuple[list[dict], int]:
    """Rows of a JSONL file written by a process that may have died mid-line. A broken *last* line is dropped
    (returns how many were dropped); a broken line in the middle means the file is corrupt -> ValueError."""
    rows, lines = [], Path(path).read_text(encoding="utf-8").split("\n")
    while lines and not lines[-1].strip():
        lines.pop()
    dropped = 0
    for i, line in enumerate(lines):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            if i == len(lines) - 1:
                dropped = 1
            else:
                raise ValueError(f"{path}: corrupt line {i + 1}")
    return rows, dropped


def atomic_write_text(path, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def atomic_write_json(path, obj) -> None:
    atomic_write_text(path, json.dumps(obj, ensure_ascii=False, indent=1, default=str))


def sha256_file(path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def config_hash(cfg: dict, ignore=()) -> str:
    """Hash of the settings that change results (shard/output/resume keys go in `ignore`)."""
    keep = {k: v for k, v in cfg.items() if k not in ignore}
    return hashlib.sha256(json.dumps(keep, sort_keys=True, default=str).encode()).hexdigest()[:16]


def shard_of(key: str, num_shards: int) -> int:
    """Stable shard index (does not depend on PYTHONHASHSEED or on the order of the inputs)."""
    if num_shards <= 1:
        return 0
    return int(hashlib.md5(key.encode()).hexdigest(), 16) % num_shards


class DurableJsonl:
    """Append-only result file of one worker. First line is a header with the run hash; every record is flushed and
    fsync'ed. On reopen: a truncated last line is dropped, the file is rewritten clean, and a header with another run
    hash is refused (never mix results of different configs). done_ids() = ids whose latest record is 'completed'."""

    def __init__(self, path, run_hash: str, metadata: dict | None = None):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.run_hash = run_hash
        self.records: dict[str, dict] = {}
        if self.path.exists() and self.path.stat().st_size:
            rows, dropped = read_jsonl_tolerant(self.path)
            header = rows[0].get("_header") if rows else None
            if not header or header.get("run_hash") != run_hash:
                raise RuntimeError(f"{self.path} belongs to run {header and header.get('run_hash')}, not {run_hash}")
            for r in rows[1:]:
                self.records[r["id"]] = r
            if dropped:
                atomic_write_text(self.path, "\n".join(dumps(r) for r in rows) + "\n")
        else:
            atomic_write_text(self.path, dumps({"_header": {"run_hash": run_hash, "created": time.time(), **(metadata or {})}}) + "\n")
        self._f = open(self.path, "a", encoding="utf-8")

    def done_ids(self) -> set[str]:
        return {i for i, r in self.records.items() if r.get("status") == "completed"}

    def append(self, rec: dict) -> None:
        previous = self.records.get(rec["id"])
        if previous and previous.get("status") == "completed":
            if previous != rec:
                raise ValueError(f"conflicting completed record: {rec['id']}")
            return
        self._f.write(dumps(rec) + "\n")
        self._f.flush()
        os.fsync(self._f.fileno())
        self.records[rec["id"]] = rec

    def close(self) -> None:
        if not self._f.closed:
            self._f.close()


def find_split_dir(split: str, roots=("/kaggle/input",), marker: str = "public_test.jsonl", max_depth: int = 6) -> Path:
    """Directory `<...>/<split>` that holds `marker` (datasets get mounted as /kaggle/input/<slug>/<split>/<split>/...)."""
    for root in map(Path, roots):
        if not root.exists():
            continue
        base_depth = len(root.parts)
        for dirpath, dirnames, filenames in os.walk(root):
            p = Path(dirpath)
            if len(p.parts) - base_depth > max_depth:
                dirnames[:] = []
                continue
            if p.name == split and marker in filenames:
                return p
    raise FileNotFoundError(f"no {split}/{marker} under {list(roots)}")


def env_info() -> dict:
    info = {"python": platform.python_version(), "platform": platform.platform(), "cpus": os.cpu_count()}
    try:
        info["gpus"] = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True, timeout=20
        ).stdout.strip().splitlines()
    except Exception:
        info["gpus"] = []
    for mod in ("torch", "transformers", "vllm", "pandas"):
        try:
            info[mod] = __import__(mod).__version__
        except Exception:
            pass
    try:
        with open("/proc/meminfo") as f:
            info["mem_total_kb"] = int(f.readline().split()[1])
    except Exception:
        pass
    return info


def set_thread_env(threads: int = 1) -> None:
    """One BLAS/OpenMP thread per process (Kaggle has few CPUs)."""
    for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        os.environ.setdefault(k, str(threads))


class Deadline:
    """Wall-clock budget of a session: stop taking new work at `stop_after_s`."""

    def __init__(self, stop_after_s: float, end_epoch=None, reserve_s=0):
        self.t0 = time.time()
        self.stop_after_s = min(stop_after_s, end_epoch - self.t0 if end_epoch is not None else stop_after_s) - reserve_s

    def elapsed(self) -> float:
        return time.time() - self.t0

    def expired(self) -> bool:
        return self.elapsed() >= self.stop_after_s

def session_deadline(cfg):
    return Deadline(float(cfg.get("session_wallclock_s", cfg.get("wallclock_s", 14400))),
                    cfg.get("session_deadline_epoch"), cfg.get("export_reserve_s", 0))


def can_start(deadline, estimate_s=0):
    return deadline.elapsed() + max(float(estimate_s), 0) < deadline.stop_after_s


class MemoryMonitor:
    """Sample this worker's nvidia-smi memory, including backend subprocesses on its visible GPU."""
    def __init__(self):
        import threading
        self.peak_mb = None
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()

    def _poll(self):
        while not self._stop.is_set():
            try:
                raw = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,used_memory", "--format=csv,noheader,nounits"],
                                     capture_output=True, text=True, timeout=3).stdout
                # Main worker + descendants (vLLM engine processes), excluding unrelated workers.
                parents = {os.getpid()}
                proc_parents = {}
                for p in Path("/proc").glob("[0-9]*/status"):
                    try:
                        fields = dict(l.split(":", 1) for l in p.read_text().splitlines() if ":" in l)
                        proc_parents[int(p.parent.name)] = int(fields["PPid"].strip())
                    except (OSError, ValueError, KeyError): pass
                while True:
                    new = {pid for pid, parent in proc_parents.items() if parent in parents} - parents
                    if not new: break
                    parents.update(new)
                used = sum(int(line.split(",")[1]) for line in raw.splitlines()
                           if line.strip() and int(line.split(",")[0]) in parents)
                self.peak_mb = max(self.peak_mb or 0, used)
            except (OSError, ValueError, subprocess.TimeoutExpired): pass
            self._stop.wait(.5)

    def close(self):
        self._stop.set()
        self._thread.join(timeout=4)
        return self.peak_mb


def wait_workers(processes, cfg):
    """Stop children at the shared work deadline, leaving the export reserve to the driver."""
    import signal
    deadline=cfg.get('session_deadline_epoch')
    end=deadline-float(cfg.get('export_reserve_s',0)) if deadline is not None else None
    codes=[]
    try:
        for process,log in processes:
            try:
                process.wait(timeout=max(.01,end-time.time()) if end is not None else None)
                codes.append(process.returncode)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid,signal.SIGKILL);process.wait();codes.append(124)
    finally:
        for process,log in processes:
            # A vLLM engine may be a descendant even after its Python coordinator exits.
            try:os.killpg(process.pid,signal.SIGKILL)
            except ProcessLookupError:pass
            if process.poll() is None:process.wait()
            log.close()
    return codes


In [ ]:
%%writefile /kaggle/working/lib/lmlib/data.py
"""LifeMate split loading, audio lists and the fixed train manifest."""
from __future__ import annotations

import hashlib
import random
import re
from collections import Counter
from pathlib import Path

from .common import find_split_dir, load_jsonl


def resolve_split_dir(cfg: dict) -> Path:
    if cfg.get("split_dir"):
        return Path(cfg["split_dir"])
    split = cfg["split"]
    return find_split_dir(split, cfg.get("input_roots", ["/kaggle/input"]), marker=f"{split}.jsonl")


def load_rows(split_dir: Path) -> list[dict]:
    return load_jsonl(split_dir / f"{split_dir.name}.jsonl")


def audio_paths(row: dict) -> list[str]:
    """Relative paths of every user audio of a sample: history turns first, then the final request."""
    out = [t["audio"] for t in row.get("history", []) if t.get("role") == "user" and t.get("audio")]
    return out + [row["audio"]]


def _template_key(row: dict) -> str:
    text = re.sub(r"\d+", "#", (row.get("audio_text") or "").lower())
    names = ",".join(sorted({c["name"] for c in (row.get("label") or {}).get("tool_calls", [])}))
    return hashlib.md5(f"{names}|{text[:40]}".encode()).hexdigest()


def label_apps(row: dict) -> set[str]:
    calls = list((row.get("label") or {}).get("tool_calls", []))
    for t in row.get("history", []):
        calls += t.get("tool_calls", []) or []
    return {c["name"].split(".", 1)[0] for c in calls if not c["name"].startswith("sys.")}


def make_manifest(rows: list[dict], seed: int = 2026, holdout_apps: int = 4, dev_frac: float = 0.12, final_frac: float = 0.12) -> dict:
    """train_fit / dev_seen / dev_unseen / final_holdout. dev_unseen = samples that use one of `holdout_apps` apps
    (drawn among apps used in 20..120 samples); no train_fit sample uses them. Near-duplicates (same tools + same
    text opening) stay in one part."""
    rng = random.Random(seed)
    use = Counter(a for r in rows for a in label_apps(r))
    cands = sorted(a for a, n in use.items() if 20 <= n <= 120)
    held = sorted(rng.sample(cands, min(holdout_apps, len(cands))))
    unseen = [r["id"] for r in rows if label_apps(r) & set(held)]
    rest = [r for r in rows if r["id"] not in set(unseen)]
    groups: dict[str, list[str]] = {}
    for r in rest:
        groups.setdefault(_template_key(r), []).append(r["id"])
    keys = sorted(groups)
    rng.shuffle(keys)
    n = len(rest)
    ids = {"train_fit": [], "dev_seen": [], "final_holdout": [], "dev_unseen": unseen}
    for k in keys:
        if len(ids["final_holdout"]) < final_frac * n:
            ids["final_holdout"] += groups[k]
        elif len(ids["dev_seen"]) < dev_frac * n:
            ids["dev_seen"] += groups[k]
        else:
            ids["train_fit"] += groups[k]
    return {"seed": seed, "holdout_apps": held, "ids": ids, "counts": {k: len(v) for k, v in ids.items()}}

def prepare_dev_run(cfg: dict, rows: list[dict]) -> dict:
    from .provenance import digest, package_hash
    ids = [r['id'] for r in rows]
    if len(ids) != len(set(ids)): raise ValueError('duplicate episode IDs')
    manifest = make_manifest(sorted(rows, key=lambda r:r['id']), seed=cfg.get('manifest_seed',2026))
    subset=cfg.get('subset','dev_seen')
    selected=sorted(set(subset) if isinstance(subset,list) else
                    (set(ids) if subset=='all' else set().union(*(manifest['ids'][s] for s in subset.split('+')))))
    if not selected:raise ValueError('selected dev manifest is empty')
    if not set(selected)<=set(ids):raise ValueError('unknown episode ID')
    for i,a in enumerate(manifest['ids']):
        for b in list(manifest['ids'])[i+1:]:
            if set(manifest['ids'][a]) & set(manifest['ids'][b]):raise ValueError('overlapping split')
    manifest.update(selected_ids=selected, input_sha256=digest(sorted(rows,key=lambda r:r['id'])),
                    heldout_ids_sha256=digest(manifest['ids']['final_holdout']),
                    code_sha256=package_hash(Path(__file__).parent))
    manifest['manifest_sha256']=digest({'ids':manifest['ids'],'selected_ids':selected,'seed':manifest['seed']})
    return manifest


def prepare_audio_manifest(rows: list[dict], episode_ids: list[str], split_dir=None) -> dict:
    from .provenance import digest
    from .common import sha256_file
    wanted=set(episode_ids);selected={r['id']:r for r in rows if r['id'] in wanted}
    if set(selected)!=wanted:raise ValueError('missing episode ID')
    roles={};gold={};sources={}
    for qid in sorted(selected):
        r=selected[qid]
        turns=[(f'history:{i}',t) for i,t in enumerate(r.get('history',[])) if t.get('role')=='user']+[('final',r)]
        for role,t in turns:
            audio=t.get('audio')
            if not audio:raise ValueError(f'missing user audio: {qid}/{role}')
            roles.setdefault(audio,[]).append({'episode_id':qid,'role':role})
            if t.get('audio_text') is not None:
                if audio in gold and gold[audio]!=t['audio_text']:raise ValueError(f'conflicting gold transcript: {audio}')
                gold[audio]=t['audio_text']
    for audio in sorted(roles):
        if split_dir is None:raise ValueError('split_dir is required to fingerprint audio bytes')
        root=Path(split_dir).resolve();p=(root/audio).resolve()
        if not p.is_relative_to(root) or not p.is_file():raise ValueError(f'missing/unsafe audio: {audio}')
        duration=None
        try:
            import soundfile as sf
            duration=sf.info(p).duration
        except (ImportError, RuntimeError):pass
        sources[audio]={'sha256':sha256_file(p),'bytes':p.stat().st_size,'duration_s':duration}
    payload={'audio_ids':sorted(roles),'episode_roles':roles,'sources':sources}
    payload['audio_manifest_sha256']=digest(payload)
    return payload


In [ ]:
%%writefile /kaggle/working/lib/lmlib/asr.py
"""Lane A: transcribe every user audio of a split into a durable cache (one record per audio path), and measure
WER / CER / entity errors on train where audio_text exists.

    python -m lmlib.asr --config cfg.json --shard I      # worker; launch() starts one per GPU
Backends: phowhisper (transformers, fp16 GPU), gipformer (sherpa-onnx RNN-T, CPU), gold (train audio_text; dry-run).
"""
from __future__ import annotations

import argparse
import json
import os
import re
import subprocess
import sys
import time
import unicodedata
from pathlib import Path

from .common import DurableJsonl, Deadline, atomic_write_json, config_hash, env_info, read_jsonl_tolerant, set_thread_env, shard_of
from .data import audio_paths, load_rows, resolve_split_dir

RUN_IGNORE = ("shard_index", "num_shards", "out_dir", "resume", "wallclock_s", "input_roots", "split_dir", "limit", "subset_ids")


# ------------------------------------------------------------------ backends
class PhoWhisper:
    def __init__(self, model="vinai/PhoWhisper-medium", revision=None, batch_size=16, **_):
        import torch
        from transformers import pipeline

        self.pipe = pipeline("automatic-speech-recognition", model=model, revision=revision, torch_dtype=torch.float16,
                             device=0 if torch.cuda.is_available() else -1)
        self.batch_size = batch_size
        self.parameter_count = sum(p.numel() for p in self.pipe.model.parameters())

    def transcribe(self, waves: list) -> list[str]:
        inputs = [{"raw": w, "sampling_rate": 16000} for w in waves]
        out = self.pipe(inputs, batch_size=self.batch_size, generate_kwargs={"do_sample": False, "num_beams": 1})
        return [o["text"].strip() for o in out]


class Gipformer:
    def __init__(self, model="g-group-ai-lab/gipformer-68M-rnnt", revision=None, int8=False, num_threads=2, **_):
        import sherpa_onnx
        from huggingface_hub import snapshot_download

        d = Path(snapshot_download(model, revision=revision, allow_patterns=["*.onnx", "tokens.txt", "config.json"]))
        from .common import sha256_file
        sfx = ".int8.onnx" if int8 else ".onnx"
        for name in ["encoder" + sfx, "decoder" + sfx, "joiner" + sfx, "tokens.txt"]:
            if not (d / name).is_file(): raise FileNotFoundError(d / name)
        self.assets = {name: sha256_file(d / name) for name in ["encoder" + sfx, "decoder" + sfx, "joiner" + sfx, "tokens.txt"]}
        for name, sha in _.get('file_hashes', {}).items():
            if name in self.assets and self.assets[name] != sha: raise ValueError(f'ASR weight checksum mismatch: {name}')
        self.rec = sherpa_onnx.OfflineRecognizer.from_transducer(
            encoder=str(d / f"encoder{sfx}"), decoder=str(d / f"decoder{sfx}"), joiner=str(d / f"joiner{sfx}"),
            tokens=str(d / "tokens.txt"), num_threads=num_threads, sample_rate=16000, feature_dim=80, decoding_method="greedy_search")

    def transcribe(self, waves: list) -> list[str]:
        streams = []
        for w in waves:
            s = self.rec.create_stream()
            s.accept_waveform(16000, w)
            streams.append(s)
        self.rec.decode_streams(streams)
        return [s.result.text.strip() for s in streams]


def make_backend(cfg: dict):
    name = cfg.get("backend", "phowhisper")
    if name == "phowhisper":
        return PhoWhisper(**cfg)
    if name == "gipformer":
        return Gipformer(**cfg)
    raise ValueError(f"unknown ASR backend {name}")


def read_wave(path: Path):
    import numpy as np
    import soundfile as sf

    w, sr = sf.read(str(path), dtype="float32")
    if w.ndim > 1:
        w = w.mean(axis=1)
    if sr != 16000:
        import librosa

        w = librosa.resample(w, orig_sr=sr, target_sr=16000)
    return np.ascontiguousarray(w)


# ------------------------------------------------------------------ metrics
def norm_text(s: str) -> str:
    t = unicodedata.normalize("NFC", s or "").lower()
    t = re.sub(r"[^\w\s]", " ", t)
    return " ".join(t.split())


def edit_distance(a: list, b: list) -> int:
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]


_NUM_WORDS = {"không", "một", "mốt", "hai", "ba", "bốn", "tư", "năm", "lăm", "sáu", "bảy", "tám", "chín", "mười", "mươi",
              "trăm", "nghìn", "ngàn", "triệu", "tỷ", "tỉ", "rưỡi", "linh", "lẻ"}
_TIME_WORDS = {"giờ", "phút", "sáng", "trưa", "chiều", "tối", "đêm", "mai", "nay", "mốt", "tuần", "tháng", "thứ", "chủ", "nhật"}


def entity_tokens(gold: str) -> dict[str, list[str]]:
    """Gold tokens that are numbers, time words, or proper names (capitalised, not sentence-initial)."""
    raw = re.findall(r"\w+", unicodedata.normalize("NFC", gold or ""))
    names = [w.lower() for i, w in enumerate(raw) if i > 0 and w[:1].isupper()]
    toks = norm_text(gold).split()
    return {
        "number": [t for t in toks if t.isdigit() or t in _NUM_WORDS],
        "time": [t for t in toks if t in _TIME_WORDS],
        "name": names,
    }


def score_pairs(pairs: list[tuple[str, str]]) -> dict:
    """pairs of (gold, hypothesis): corpus WER/CER and per-entity-class token recall."""
    we = wn = ce = cn = 0
    ent = {k: [0, 0] for k in ("number", "time", "name")}
    for g, h in pairs:
        gt, ht = norm_text(g).split(), norm_text(h).split()
        we += edit_distance(gt, ht)
        wn += len(gt)
        gc, hc = list(norm_text(g).replace(" ", "")), list(norm_text(h).replace(" ", ""))
        ce += edit_distance(gc, hc)
        cn += len(gc)
        hset = set(ht)
        for k, toks in entity_tokens(g).items():
            ent[k][0] += sum(t in hset for t in toks)
            ent[k][1] += len(toks)
    return {
        "n": len(pairs),
        "wer": we / wn if wn else None,
        "word_edits": we, "gold_words": wn,
        "cer": ce / cn if cn else None,
        "char_edits": ce, "gold_chars": cn,
        "entity_recall": {k: (a / b if b else None) for k, (a, b) in ent.items()},
        "entity_counts": {k: b for k, (a, b) in ent.items()},
    }


def gold_texts(rows: list[dict]) -> dict[str, str]:
    out = {}
    for r in rows:
        for t in r.get("history", []):
            if t.get("role") == "user" and t.get("audio") and t.get("audio_text") is not None:
                if t["audio"] in out and out[t["audio"]] != t["audio_text"]: raise ValueError("conflicting gold transcript")
                out[t["audio"]] = t["audio_text"]
        if r.get("audio_text") is not None:
            if r["audio"] in out and out[r["audio"]] != r["audio_text"]: raise ValueError("conflicting gold transcript")
            out[r["audio"]] = r["audio_text"]
    return out


# ------------------------------------------------------------------ worker
def run_hash(cfg: dict) -> str:
    return config_hash(cfg, ignore=RUN_IGNORE)


def cache_name(cfg: dict) -> str:
    return f"asr_{cfg['split']}_{cfg['asr']['backend']}"


def legacy_work(cfg: dict, shard_index: int) -> dict:
    set_thread_env(int(cfg.get("cpu_threads", 1)))
    deadline = Deadline(float(cfg.get("wallclock_s", 3.5 * 3600)))
    split_dir = resolve_split_dir(cfg)
    rows = load_rows(split_dir)
    if cfg.get("subset_ids"):
        keep = set(cfg["subset_ids"])
        rows = [r for r in rows if r["id"] in keep]
    if cfg.get("limit"):
        rows = rows[: int(cfg["limit"])]
    paths = sorted({p for r in rows for p in audio_paths(r)})
    n = int(cfg.get("num_shards", 1))
    mine = [p for p in paths if shard_of(p, n) == shard_index]
    out = Path(cfg["out_dir"])
    cache = DurableJsonl(out / f"{cache_name(cfg)}.shard{shard_index}.jsonl", run_hash(cfg))
    todo = [p for p in mine if p not in cache.done_ids()]
    t0 = time.time()
    backend = None
    gold = gold_texts(rows) if cfg["asr"]["backend"] == "gold" else None
    if gold is None:
        backend = make_backend(cfg["asr"])
    load_s = time.time() - t0
    bs = int(cfg["asr"].get("batch_size", 16))
    audio_s = 0.0
    t_run = time.time()
    stopped = False
    for b in range(0, len(todo), bs):
        if deadline.expired():
            stopped = True
            break
        chunk = todo[b : b + bs]
        try:
            if gold is not None:
                texts, secs = [gold.get(p, "") for p in chunk], [0.0] * len(chunk)
            else:
                waves = [read_wave(split_dir / p) for p in chunk]
                secs = [len(w) / 16000 for w in waves]
                tb = time.time()
                texts = backend.transcribe(waves)
                per = (time.time() - tb) / len(chunk)
            for p, t, s in zip(chunk, texts, secs):
                cache.append({"id": p, "status": "completed", "text": t, "audio_s": round(s, 2), "asr_s": round(per, 3) if gold is None else 0})
                audio_s += s
        except Exception as e:
            for p in chunk:
                cache.append({"id": p, "status": "retryable", "error": f"{type(e).__name__}: {e}"})
        print(f"[asr shard {shard_index}] {b + len(chunk)}/{len(todo)}", flush=True)
    cache.close()
    peak = None
    try:
        import torch

        peak = round(torch.cuda.max_memory_allocated() / 2**30, 2) if torch.cuda.is_available() else None
    except Exception:
        pass
    summary = {"shard": shard_index, "audios_in_shard": len(mine), "todo": len(todo), "stopped_early_by_wallclock": stopped,
               "model_load_s": round(load_s, 1), "run_s": round(time.time() - t_run, 1), "audio_s": round(audio_s, 1),
               "rtf": round((time.time() - t_run) / max(audio_s, 1e-9), 4), "peak_vram_gb": peak, "env": env_info()}
    atomic_write_json(out / f"{cache_name(cfg)}.summary.shard{shard_index}.json", summary)
    return summary


def load_cache(dirs, split: str | None = None, backend: str | None = None) -> dict[str, str]:
    """{audio relative path: text} from every asr_*.shard*.jsonl under the given dirs (output of this notebook, or
    an attached output version)."""
    texts = {}
    pat = f"asr_{split or '*'}_{backend or '*'}.shard*.jsonl"
    for d in dirs:
        for p in sorted(Path(d).rglob(pat)):
            rows, _ = read_jsonl_tolerant(p)
            for r in rows[1:]:
                if r.get("status") == "completed":
                    texts[r["id"]] = r["text"]
    return texts


def evaluate(cfg: dict) -> dict:
    split_dir = resolve_split_dir(cfg)
    rows = load_rows(split_dir)
    gold = gold_texts(rows)
    hyp = load_cache([cfg["out_dir"]], cfg["split"], cfg["asr"]["backend"])
    pairs = [(gold[p], hyp[p]) for p in sorted(gold) if p in hyp]
    rep = score_pairs(pairs)
    worst = sorted(((edit_distance(norm_text(g).split(), norm_text(h).split()) / max(len(norm_text(g).split()), 1), p, g, h)
                    for p, (g, h) in zip([p for p in sorted(gold) if p in hyp], pairs)), reverse=True)[:30]
    rep["worst"] = [{"audio": p, "wer": round(w, 3), "gold": g, "hyp": h} for w, p, g, h in worst]
    atomic_write_json(Path(cfg["out_dir"]) / f"{cache_name(cfg)}.metrics.json", rep)
    return rep


def launch(cfg_path: str, cfg: dict, gpus: list[int], lib_root: str, module: str = "lmlib.asr", python: str | None = None) -> list[int]:
    out = Path(cfg["out_dir"])
    out.mkdir(parents=True, exist_ok=True)
    procs = []
    for i in range(int(cfg.get("num_shards", 1))):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpus[i % len(gpus)]) if gpus else "", PYTHONUNBUFFERED="1",
                   PYTHONPATH=lib_root + os.pathsep + os.environ.get("PYTHONPATH", ""))
        log = open(out / f"{module.split('.')[-1]}_worker{i}.log", "a", encoding="utf-8")
        procs.append((subprocess.Popen([python or sys.executable, "-m", module, "--config", cfg_path, "--shard", str(i)], env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True), log))
    from .common import wait_workers
    return wait_workers(procs,cfg)



# Strict dev cache contract. Legacy public/SFT caches remain readable only in their legacy presets.
def asr_components(backend):
    from .provenance import source_hash
    from .data import prepare_audio_manifest
    cls = PhoWhisper if backend == 'phowhisper' else Gipformer
    return {'read_wave':source_hash(read_wave), 'backend':source_hash(cls),
            'audio_selection':source_hash(prepare_audio_manifest), 'worker':source_hash(work)}


def asr_cache_key(manifest, backend_cfg, component_hashes):
    from .provenance import digest, RUNTIME_KEYS
    return digest({'audio_manifest_sha256':manifest['audio_manifest_sha256'],
                   'backend_config':{k:v for k,v in backend_cfg.items() if k not in RUNTIME_KEYS},
                   'components':component_hashes, 'preprocess':'mono-float32-16khz-v1'})


def cache_expectation(manifest, backend_cfg, execution=None):
    components=asr_components(backend_cfg['backend'])
    return {'schema':'asr_cache_v1','content_key':asr_cache_key(manifest,backend_cfg,components),
            'audio_manifest_sha256':manifest['audio_manifest_sha256'], 'backend':backend_cfg['backend'],
            'model':backend_cfg['model'],'revision':backend_cfg['revision'],
            'decoding':backend_cfg,'preprocess':'mono-float32-16khz-v1','component_hashes':components,
            'source_execution':execution}


def load_verified_transcripts(cache_dir, expected, audio_ids, allow_partial=False):
    from .provenance import collect_shards
    root=Path(cache_dir);p=root/'cache_manifest.json'
    if not p.is_file():raise ValueError(f'no verified ASR cache manifest: {root}')
    got=json.loads(p.read_text())
    for k in ('schema','content_key','audio_manifest_sha256','backend'):
        if not expected.get(k) or got.get(k)!=expected[k]:raise ValueError(f'ASR cache mismatch: {k}')
    records=collect_shards(root,'transcripts.shard*.jsonl',allowed_ids=audio_ids)
    for f in root.glob('transcripts.shard*.jsonl'):
        rows,_=read_jsonl_tolerant(f);h=rows[0]['_header']
        if h.get('content_key',h.get('run_hash'))!=expected['content_key']:raise ValueError('ASR shard content mismatch')
    texts={i:r['text'] for i,r in records.items() if r.get('status')=='completed' and isinstance(r.get('text'),str)}
    missing=set(audio_ids)-set(texts)
    if missing and not allow_partial:raise ValueError(f'missing ASR audio, including history: {sorted(missing)[:8]}')
    return texts


def work(cfg, shard_index, backend=None):
    if not cfg.get('strict_dev'):return legacy_work(cfg,shard_index)
    from .common import session_deadline, can_start, MemoryMonitor
    from .provenance import digest, freeze_json
    from .data import prepare_audio_manifest
    set_thread_env(int(cfg.get('cpu_threads',1)))
    deadline=session_deadline(cfg);monitor=MemoryMonitor();out=Path(cfg['out_dir'])
    sd=resolve_split_dir(cfg);rows=load_rows(sd)
    manifest=prepare_audio_manifest(rows,cfg['subset_ids'],sd)
    expected=cache_expectation(manifest,cfg['asr'],cfg.get('execution_id'))
    # Source execution is provenance, not a semantic constraint when resuming.
    manifest_path=out/'cache_manifest.json'
    if manifest_path.exists():
        previous=json.loads(manifest_path.read_text())
        if previous['content_key']!=expected['content_key']:raise ValueError('ASR cache changed')
    else:atomic_write_json(manifest_path,expected)
    freeze_json(out/'audio_manifest.json',manifest)
    n=int(cfg['num_shards']);mine=[p for p in manifest['audio_ids'] if shard_of(p,n)==shard_index]
    rh=digest({'content_key':expected['content_key'],'num_shards':n})
    cache=DurableJsonl(out/f'transcripts.shard{shard_index}.jsonl',rh,expected)
    todo=[p for p in mine if p not in cache.done_ids()]
    start=time.time();loaded_s=0.;audio_s=0.;completed=0;preprocess_s=decode_s=0.;batch_est=float(cfg.get('batch_estimate_s',30))
    try:
        if todo and can_start(deadline,batch_est):
            t=time.time();backend=backend or make_backend(cfg['asr']);loaded_s=time.time()-t
        smoke = None
        if backend is not None and cfg.get('smoke_audio_ids'):
            waves = [read_wave(sd / p) for p in cfg['smoke_audio_ids']]
            single = []
            for wave in waves:
                if not can_start(deadline,batch_est): raise RuntimeError('deadline reached during ASR smoke')
                began = time.time(); result = backend.transcribe([wave]); single.append(time.time()-began)
                if len(result) != 1: raise RuntimeError('ASR batch=1 smoke failed')
            began = time.time(); result = backend.transcribe(waves); batch_s = time.time()-began
            if len(result) != len(waves): raise RuntimeError('ASR batch=16 smoke failed')
            smoke = {'n':len(waves),'batch1_s':single,'batch16_s':batch_s,'metric_used_for_selection':False}
            batch_est = max(batch_est,batch_s*1.5)
        if getattr(backend,'parameter_count',0) + 4022468096 >= 6000000000:
            raise ValueError('LifeMate total parameter limit exceeded')
        for b in range(0,len(todo),int(cfg['asr'].get('batch_size',16))):
            if backend is None or not can_start(deadline,batch_est):break
            chunk=todo[b:b+int(cfg['asr'].get('batch_size',16))];t=time.time()
            try:
                waves=[read_wave(sd/p) for p in chunk];preprocess_s+=time.time()-t;tg=time.time()
                texts=backend.transcribe(waves);elapsed=time.time()-tg;decode_s+=elapsed
                if len(texts)!=len(chunk) or any(not isinstance(t,str) for t in texts):raise ValueError('ASR result cardinality/type mismatch')
                for p,text,w in zip(chunk,texts,waves):
                    duration=len(w)/16000;audio_s+=duration;completed+=1
                    cache.append({'id':p,'status':'completed','text':text,'audio_s':duration,
                                  'batch_decode_s':elapsed,'batch_size':len(chunk),'execution_id':cfg.get('execution_id')})
                batch_est=max(batch_est,(time.time()-t)*1.25)
            except Exception as e:
                for p in chunk:cache.append({'id':p,'status':'retryable','error':f'{type(e).__name__}: {e}'})
        summary={'shard':shard_index,'run_hash':rh,'expected':len(mine),'completed':len(cache.done_ids()),
                 'pending':len(set(mine)-cache.done_ids()),'model_load_s':loaded_s,'preprocess_s':preprocess_s,
                 'decode_s':decode_s,'run_s':time.time()-start,'audio_s':audio_s,
                 's_per_sample':(time.time()-start)/completed if completed else None,
                 'rtf':(time.time()-start)/audio_s if audio_s else None,
                 'throughput_audio_s_per_s':audio_s/(time.time()-start) if completed else None,
                 'peak_vram_mb':monitor.close(),'smoke':smoke,'parameter_count':getattr(backend,'parameter_count',None),'assets':getattr(backend,'assets',None),'execution_id':cfg.get('execution_id')}
        atomic_write_json(out/f'summary.shard{shard_index}.json',summary)
        return summary
    finally:cache.close();monitor.close()


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--shard", type=int, default=0)
    a = ap.parse_args(argv)
    cfg = json.loads(Path(a.config).read_text(encoding="utf-8"))
    print(json.dumps(work(cfg, a.shard), ensure_ascii=False, indent=1))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/lib/lmlib/agent.py
"""LifeMate agent loop: transcript + context + history (with tool_results) + installed-app schemas -> the LLM emits
one JSON action per step -> schema check (env.validate) -> env.call -> result goes into the next step.

Stops on {"done": true}, after sys.ask_missing_info, on a repeated identical call, after `max_invalid` rejected
proposals, at `max_steps`, or when the per-sample time budget runs out. The budget counts the sample's own share of
generation time (llm_s) + ASR, not wall-clock: samples are stepped together, so one batched step takes minutes on a
T4 and a wall-clock budget cut every sample after its first call (lm-B-004). The A30 SLA is measured separately. Every env.call is final (one ToolEnv per
sample); invalid proposals are never sent to env.call. Samples are stepped in batches so vLLM can batch them.
"""
from __future__ import annotations

import json
import re
import time
from dataclasses import dataclass, field

SYSTEM = """Bạn là trợ lý ảo điều khiển các ứng dụng trên điện thoại của người dùng Việt Nam bằng cách gọi tool.
Mỗi lượt bạn chỉ trả về ĐÚNG MỘT đối tượng JSON, không giải thích:
- gọi tool:   {"name": "<app>.<tool>", "arguments": {...}}
- kết thúc:   {"done": true}

Quy tắc:
1. Chỉ gọi tool có trong DANH SÁCH TOOL bên dưới (thuộc app đã cài) hoặc sys.ask_missing_info. Đúng tên tham số, kiểu, enum.
2. Không bao giờ bịa ID (contact_id, device_id, event_id...). Muốn có ID thì gọi tool đọc (read) trước rồi lấy ID từ kết quả.
3. Chỉ gọi tool đọc khi thật sự cần cho yêu cầu cuối. Nếu lịch sử (tool_results) đã có thông tin trả lời được thì dùng lại, không gọi lại.
4. Thiếu thông tin bắt buộc mà người dùng phải tự nói (giờ, số tiền, người nhận...) và không suy ra được, hoặc kết quả
   tìm kiếm mơ hồ (nhiều người/đồ vật khớp): gọi
   {"name": "sys.ask_missing_info", "arguments": {"tool_name": "<tool ghi cần gọi>", "params": ["<tham số thiếu>"]}} rồi dừng.
   Không hỏi tham số của tool đọc (search/list/get): cứ gọi tool đọc với thông tin đã có, bỏ tham số tuỳ chọn.
   Không hỏi ID: ID lấy từ kết quả tool đọc.
5. Yêu cầu ngoài khả năng của các app đã cài, tán gẫu, người dùng tự huỷ ("thôi khỏi"), hoặc yêu cầu xấu: trả về {"done": true} ngay, không gọi tool nào.
6. Thời gian dạng ISO 8601 có múi giờ +07:00 (vd 2026-11-28T07:30:00+07:00), ngày dạng YYYY-MM-DD; tính "mai", "tối nay", "thứ sáu tuần sau"... theo current_time.
7. Tham số tìm kiếm (query, tên người...) dùng đúng cụm từ người dùng nói. Văn bản tự do (nội dung tin nhắn, tiêu đề, ghi chú) viết ngắn gọn đúng ý người dùng.
8. Làm đủ mọi việc người dùng yêu cầu ở lượt cuối (có thể nhiều tool liên tiếp), xong thì {"done": true}.
9. Phần VÍ DỤ TƯƠNG TỰ (nếu có) là các yêu cầu khác đã làm đúng: học cách chọn tool, thứ tự bước, cách ghi tham số.
   ID và dữ liệu trong ví dụ thuộc người dùng khác, không dùng cho yêu cầu này.
"""


def _short(v, n: int) -> str:
    s = json.dumps(v, ensure_ascii=False, separators=(",", ":"))
    return s if len(s) <= n else s[: n - 1] + "…"


def _param_spec(name: str, p: dict, required: set) -> str:
    t = p.get("type", "")
    if t == "array":
        t = f"array<{(p.get('items') or {}).get('type', '')}>"
    bits = [f"{name}{'*' if name in required else ''}:{t}"]
    if p.get("enum"):
        bits.append("enum=" + "|".join(map(str, p["enum"])))
    if p.get("format"):
        bits.append(f"format={p['format']}")
    if "minimum" in p or "maximum" in p:
        bits.append(f"range=[{p.get('minimum', '')},{p.get('maximum', '')}]")
    if p.get("x-ref"):
        bits.append(f"id từ {p['x-ref']}")
    if p.get("x-match") == "fuzzy":
        bits.append("văn bản tự do")
    if p.get("description"):
        bits.append(p["description"][:80])
    return " ".join(bits)


class Catalog:
    def __init__(self, catalog: dict):
        self.tools = {}
        self.app_desc = {}
        for app in catalog["apps"]:
            self.app_desc[app["id"]] = f"{app.get('name_vi', '')}: {app.get('description', '')}"
            for t in app["tools"]:
                self.tools[t["name"]] = t
        self._render: dict[str, str] = {}

    def fuzzy_params(self, name: str) -> set[str]:
        props = ((self.tools.get(name) or {}).get("parameters") or {}).get("properties", {})
        return {k for k, v in props.items() if v.get("x-match") == "fuzzy"}

    def render_tool(self, name: str) -> str:
        if name not in self._render:
            t = self.tools[name]
            ps = t.get("parameters") or {}
            req = set(ps.get("required", []))
            params = "; ".join(_param_spec(k, v, req) for k, v in (ps.get("properties") or {}).items()) or "không tham số"
            one = ps.get("x-one-of")
            extra = f" | bắt buộc một trong: {one}" if one else ""
            self._render[name] = f"- {name} [{t.get('kind')}] {t.get('description', '')} | {params}{extra}"
        return self._render[name]

    def render_apps(self, installed: dict) -> str:
        lines = []
        for app, a in installed.items():
            lines.append(f"## {app} — {self.app_desc.get(app, '')}")
            lines += [self.render_tool(n) for n in a.get("tools", []) if n in self.tools]
        lines.append("## sys")
        if "sys.ask_missing_info" in self.tools:
            lines.append(self.render_tool("sys.ask_missing_info"))
        return "\n".join(lines)


def render_history(row: dict, transcripts: dict[str, str], result_chars: int) -> str:
    out = []
    for t in row.get("history", []):
        if t.get("role") == "user":
            out.append(f"Người dùng: {transcripts.get(t.get('audio'), t.get('audio_text') or '(không nghe rõ)')}")
        else:
            out.append(f"Trợ lý: {t.get('text', '')}")
            for c, r in zip(t.get("tool_calls") or [], (t.get("tool_results") or []) + [None] * 10):
                out.append(f"  đã gọi {_short(c, 400)}")
                if r is not None:
                    out.append(f"  kết quả {_short(r, result_chars)}")
    return "\n".join(out) or "(không có)"


def build_messages(row: dict, request_text: str, catalog: Catalog, transcripts: dict, steps: list[dict], result_chars: int = 1500,
                   examples: str = "") -> list[dict]:
    ctx = row["context"]
    head = {k: ctx.get(k) for k in ("current_time", "timezone", "location", "user_profile")}
    parts = [
        "# NGỮ CẢNH\n" + json.dumps(head, ensure_ascii=False),
        "# DANH SÁCH TOOL (app đã cài)\n" + catalog.render_apps(ctx.get("installed_apps", {})),
        *(["# VÍ DỤ TƯƠNG TỰ (người dùng khác)\n" + examples] if examples else []),
        "# LỊCH SỬ HỘI THOẠI\n" + render_history(row, transcripts, result_chars),
        f"# YÊU CẦU CUỐI CỦA NGƯỜI DÙNG\n{request_text}",
    ]
    if steps:
        lines = []
        for i, s in enumerate(steps, 1):
            if s["kind"] == "call":
                lines.append(f"{i}. đã gọi {_short({'name': s['name'], 'arguments': s['arguments']}, 600)}\n   kết quả {_short(s['result'], result_chars)}")
            else:
                label = "hệ thống nhắc" if s.get("guard") else "đề xuất bị từ chối"
                lines.append(f"{i}. {label} (không gọi): {_short(s.get('proposal'), 300)} — lỗi: {s['error']}")
        parts.append("# CÁC BƯỚC ĐÃ LÀM CHO YÊU CẦU CUỐI\n" + "\n".join(lines))
    parts.append("Trả về JSON cho bước tiếp theo:")
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "\n\n".join(parts)}]


def _first_json_object(text: str):
    text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S)
    m = re.search(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", text, re.S)
    if m:
        text = m.group(1)
    start = text.find("{")
    while start != -1:
        depth, in_str, esc = 0, False, False
        for i in range(start, len(text)):
            ch = text[i]
            if in_str:
                esc = (ch == "\\") and not esc
                if ch == '"' and not esc:
                    in_str = False
                continue
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[start : i + 1])
                    except json.JSONDecodeError:
                        break
        start = text.find("{", start + 1)
    return None


def parse_action(text: str):
    """('call', name, args) | ('done', None, None) | ('invalid', None, reason)."""
    obj = _first_json_object(text)
    if not isinstance(obj, dict):
        return ("invalid", None, "không đọc được JSON")
    if obj.get("done") is True or obj.get("name") in ("done", "finish"):
        return ("done", None, None)
    name = obj.get("name") or obj.get("tool")
    args = obj.get("arguments", obj.get("args", obj.get("parameters", {})))
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except json.JSONDecodeError:
            return ("invalid", None, "arguments không phải JSON object")
    if not isinstance(name, str) or not isinstance(args, dict):
        return ("invalid", None, "thiếu name hoặc arguments")
    return ("call", name, args)


@dataclass
class AgentConfig:
    max_steps: int = 8
    max_new_tokens: int = 256
    max_invalid: int = 2
    sample_budget_s: float = 30.0
    result_chars: int = 1500
    budget_mode: str = "amortized"  # legacy public proxy; strict paired-dev uses admitted wall time
    block_read_ask: bool = True  # reject sys.ask_missing_info aimed at a read tool (gold: 210/217 asks target write tools)
    # Guards (lm-B-015). A guard refuses one proposal and feeds the reason back as a step; after `max_guard` refusals
    # the sample falls back to the plain behaviour, so a guard can delay the model's final choice but never block it.
    block_xref_ask: bool = False  # ask for an x-ref (ID) param before any read call here or in the history
    read_repeat_feedback: bool = False  # identical repeated read call -> refused, pointing at the earlier result
    nudge_done_after_read: bool = False  # first done after read calls only (no write): re-check the request once
    # call with an x-ref (ID) value that no earlier tool result (here or in the history) contains. Gold train: 1277/1278
    # x-ref args are grounded; lm-sft-016 dev: 20 samples use an unseen ID, 18 of them fail.
    ground_xref: bool = False
    max_guard: int = 3


@dataclass
class Episode:
    row: dict
    request: str
    env: object
    t0: float
    asr_s: float = 0.0
    steps: list = field(default_factory=list)
    invalid: int = 0
    finished: str | None = None  # done | ask | repeat | max_steps | invalid | budget | too_long
    gen_tokens: int = 0
    llm_s: float = 0.0
    examples: str = ""  # rendered few-shot block (lmlib.fewshot), fixed for the episode
    guard_rejects: int = 0
    guards: dict = field(default_factory=dict)  # guard name -> times it refused a proposal


NUDGE_DONE = ("bạn mới chỉ gọi tool đọc, chưa làm thao tác nào. Đọc lại YÊU CẦU CUỐI: người dùng muốn làm gì (gửi, đặt, "
              "tạo, bật/tắt, chuyển tiền, huỷ...) thì gọi tool ghi tương ứng, thiếu thông tin thì sys.ask_missing_info; "
              "chỉ hỏi thông tin hoặc không cần làm gì thêm thì trả lại {\"done\": true}")


def _kind(catalog: Catalog, name) -> str | None:
    return (catalog.tools.get(name) or {}).get("kind")


def _only_reads(e: Episode, catalog: Catalog) -> bool:
    calls = [s["name"] for s in e.steps if s["kind"] == "call"]
    return bool(calls) and all(_kind(catalog, n) == "read" for n in calls)


def _read_seen(e: Episode, catalog: Catalog) -> bool:
    """A read call in this request or in the history. Gold x-ref asks (train): 114/131 come after a read in the
    request, the other 17 after one in the history; none without any."""
    if any(s["kind"] == "call" and _kind(catalog, s["name"]) == "read" for s in e.steps):
        return True
    return any(_kind(catalog, c.get("name")) == "read" for t in e.row.get("history", []) if t.get("role") == "assistant"
               for c in t.get("tool_calls") or [] if isinstance(c, dict))


def _xref_params(catalog: Catalog, tool_name, params) -> dict:
    props = ((catalog.tools.get(tool_name) or {}).get("parameters") or {}).get("properties", {})
    return {p: props[p]["x-ref"] for p in params if isinstance(p, str) and (props.get(p) or {}).get("x-ref")}


def _scalars(x, out: set) -> set:
    if isinstance(x, dict):
        for v in x.values():
            _scalars(v, out)
    elif isinstance(x, list):
        for v in x:
            _scalars(v, out)
    elif x is not None:
        out.add(str(x))
    return out


def _seen_values(e: Episode) -> set:
    """Every scalar in the tool results of the history and of the calls made so far for this request."""
    seen = set()
    for t in e.row.get("history", []):
        if t.get("role") == "assistant":
            _scalars(t.get("tool_results") or [], seen)
    for s in e.steps:
        if s["kind"] == "call":
            _scalars(s.get("result"), seen)
    return seen


def _read_hint(e: Episode, catalog: Catalog, refs) -> str:
    installed = e.row["context"].get("installed_apps", {})
    reads = [n for a in dict.fromkeys(str(r).split(".")[0] for r in refs)
             for n in (installed.get(a) or {}).get("tools", []) if _kind(catalog, n) == "read"][:4]
    return f" (vd {', '.join(reads)})" if reads else ""


def _guard(e: Episode, key: str, proposal, msg: str) -> None:
    e.steps.append({"kind": "invalid", "guard": key, "proposal": proposal, "error": msg})
    e.guard_rejects += 1
    e.guards[key] = e.guards.get(key, 0) + 1


def run_episodes(episodes: list[Episode], llm, catalog: Catalog, transcripts: dict, cfg: AgentConfig) -> None:
    for step in range(cfg.max_steps + cfg.max_invalid + cfg.max_guard + 1):
        active = [e for e in episodes if e.finished is None]
        if not active:
            return
        for e in active:
            n_calls = sum(s["kind"] == "call" for s in e.steps)
            if n_calls >= cfg.max_steps:
                e.finished = "max_steps"
            elif (time.time() - e.t0 if cfg.budget_mode == "wall" else e.llm_s + e.asr_s) > cfg.sample_budget_s:
                e.finished = "budget"
        active = [e for e in active if e.finished is None]
        if not active:
            return
        convs = [build_messages(e.row, e.request, catalog, transcripts, e.steps, cfg.result_chars, e.examples) for e in active]
        t = time.time()
        outs = llm.generate(convs, cfg.max_new_tokens)
        if len(outs) != len(active) or any(o.finish == "error" for o in outs):
            raise RuntimeError("LLM backend failed or returned the wrong number of outputs")
        dt = (time.time() - t) / len(active)
        for e, o in zip(active, outs):
            e.llm_s += dt
            e.gen_tokens += o.output_tokens
            if o.finish == "too_long":
                e.finished = "too_long"
                continue
            kind, name, args = parse_action(o.text)
            if kind == "done":
                if (cfg.nudge_done_after_read and not e.guards.get("nudge_done") and e.guard_rejects < cfg.max_guard
                        and _only_reads(e, catalog)):
                    _guard(e, "nudge_done", {"done": True}, NUDGE_DONE)
                    continue
                e.finished = "done"
                continue
            if kind == "invalid":
                e.steps.append({"kind": "invalid", "proposal": o.text[:300], "error": args})
                e.invalid += 1
                if e.invalid >= cfg.max_invalid:
                    e.finished = "invalid"
                continue
            same = [i for i, s in enumerate(e.steps, 1) if s["kind"] == "call" and s["name"] == name and s["arguments"] == args]
            if same:
                if cfg.read_repeat_feedback and _kind(catalog, name) == "read" and e.guard_rejects < cfg.max_guard:
                    _guard(e, "repeat_read", {"name": name, "arguments": args},
                           f"đã gọi đúng lệnh này ở bước {same[0]}, kết quả ở trên: dùng lại kết quả đó, làm bước tiếp theo "
                           "hoặc trả {\"done\": true}")
                    continue
                e.finished = "repeat"
                continue
            if cfg.block_read_ask and name == "sys.ask_missing_info" and (catalog.tools.get(args.get("tool_name")) or {}).get("kind") == "read":
                e.steps.append({"kind": "invalid", "proposal": {"name": name, "arguments": args},
                                "error": "không hỏi tham số của tool đọc: hãy tự gọi tool đọc với thông tin đã có"})
                e.invalid += 1
                if e.invalid >= cfg.max_invalid:
                    e.finished = "invalid"
                continue
            if cfg.block_xref_ask and name == "sys.ask_missing_info" and e.guard_rejects < cfg.max_guard:
                asked = args.get("params") if isinstance(args.get("params"), list) else []
                refs = _xref_params(catalog, args.get("tool_name"), asked)
                if refs and not _read_seen(e, catalog):
                    installed = e.row["context"].get("installed_apps", {})
                    reads = [n for a in dict.fromkeys(str(r).split(".")[0] for r in refs.values())
                             for n in (installed.get(a) or {}).get("tools", []) if _kind(catalog, n) == "read"][:4]
                    hint = f" (vd {', '.join(reads)})" if reads else ""
                    _guard(e, "xref_ask", {"name": name, "arguments": args},
                           f"không hỏi {', '.join(refs)}: đó là ID, lấy từ kết quả tool đọc. Gọi tool đọc trước{hint} với "
                           "thông tin người dùng đã nói; chỉ hỏi lại khi không tìm thấy hoặc có nhiều kết quả khớp")
                    continue
            if cfg.ground_xref and isinstance(args, dict) and not str(name).startswith("sys.") and e.guard_rejects < cfg.max_guard:
                props = ((catalog.tools.get(name) or {}).get("parameters") or {}).get("properties", {})
                seen = _seen_values(e)
                bad = {p: v for p, v in args.items() if (props.get(p) or {}).get("x-ref")
                       and any(str(x) not in seen for x in (v if isinstance(v, list) else [v]))}
                if bad:
                    hint = _read_hint(e, catalog, [props[p]["x-ref"] for p in bad])
                    _guard(e, "ground_xref", {"name": name, "arguments": args},
                           f"{', '.join(f'{p}={v}' for p, v in bad.items())} không có trong kết quả tool nào: ID phải lấy từ "
                           f"kết quả tool đọc. Gọi tool đọc trước{hint} rồi dùng đúng ID trong kết quả")
                    continue
            chk = e.env.validate(name, args)
            if not chk.get("ok"):
                e.steps.append({"kind": "invalid", "proposal": {"name": name, "arguments": args}, "error": chk.get("error")})
                e.invalid += 1
                if e.invalid >= cfg.max_invalid:
                    e.finished = "invalid"
                continue
            result = e.env.call(name, args)
            e.steps.append({"kind": "call", "name": name, "arguments": args, "result": result})
            if name == "sys.ask_missing_info":
                e.finished = "ask"
    for e in episodes:
        if e.finished is None:
            e.finished = "max_steps"

In [ ]:
%%writefile /kaggle/working/lib/lmlib/llm.py
"""Greedy chat generation behind one interface: vLLM (fast, Kaggle T4 in fp16), HF transformers (fallback) or a mock.

Prompts are rendered with the model's own chat template (enable_thinking passed when the template knows it) and
prompts that do not fit `max_model_len - max_tokens` are not generated but returned with finish='too_long'.
"""
from __future__ import annotations

import os
import random
from dataclasses import dataclass
from typing import Callable


@dataclass
class GenOut:
    text: str
    prompt_tokens: int
    output_tokens: int
    finish: str  # stop | length | too_long | error


def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np

        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


class BaseLLM:
    max_model_len: int = 16384

    def render(self, conv: list[dict]) -> str:
        raise NotImplementedError

    def count_tokens(self, conv: list[dict]) -> int:
        raise NotImplementedError

    def generate(self, convs: list[list[dict]], max_tokens: int) -> list[GenOut]:
        raise NotImplementedError


def _apply_template(tok, conv, thinking: bool) -> str:
    try:
        return tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True, enable_thinking=thinking)
    except TypeError:
        return tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)


class VLLM(BaseLLM):
    def __init__(self, model: str, revision: str | None = None, dtype: str = "half", max_model_len: int = 16384,
                 gpu_memory_utilization: float = 0.90, quantization: str | None = None, seed: int = 0,
                 enable_thinking: bool = False, tensor_parallel_size: int = 1, enforce_eager: bool = False,
                 max_num_seqs: int | None = None, max_num_batched_tokens: int | None = None,
                 enable_chunked_prefill: bool | None = None, language_model_only: bool | None = None, **_):
        from vllm import LLM, SamplingParams  # noqa: F401

        self._SamplingParams = SamplingParams
        scheduling = {k:v for k,v in {"max_num_seqs":max_num_seqs,"max_num_batched_tokens":max_num_batched_tokens,
                      "enable_chunked_prefill":enable_chunked_prefill,
                      "language_model_only":language_model_only}.items() if v is not None}
        self.llm = LLM(model=model, revision=revision, dtype=dtype, max_model_len=max_model_len, seed=seed,
                       gpu_memory_utilization=gpu_memory_utilization, quantization=quantization,
                       tensor_parallel_size=tensor_parallel_size, enable_prefix_caching=True, enforce_eager=enforce_eager,
                       **scheduling)
        self.tok = self.llm.get_tokenizer()
        self.max_model_len = max_model_len
        self.thinking = enable_thinking

    def render(self, conv):
        return _apply_template(self.tok, conv, self.thinking)

    def count_tokens(self, conv):
        return len(self.tok(self.render(conv), add_special_tokens=False)["input_ids"])

    def generate(self, convs, max_tokens):
        prompts = [self.render(c) for c in convs]
        lens = [len(self.tok(p, add_special_tokens=False)["input_ids"]) for p in prompts]
        todo = [i for i, n in enumerate(lens) if n + max_tokens <= self.max_model_len]
        outs = [GenOut("", n, 0, "too_long") for n in lens]
        if todo:
            sp = self._SamplingParams(temperature=0.0, top_p=1.0, max_tokens=max_tokens, seed=0)
            res = self.llm.generate([prompts[i] for i in todo], sp, use_tqdm=False)
            for i, r in zip(todo, res):
                o = r.outputs[0]
                outs[i] = GenOut(o.text, lens[i], len(o.token_ids), "length" if o.finish_reason == "length" else "stop")
        return outs


class HFLLM(BaseLLM):
    def __init__(self, model: str, revision: str | None = None, dtype: str = "half", max_model_len: int = 16384,
                 load_in_4bit: bool = False, enable_thinking: bool = False, batch_size: int = 4, device_map="auto", **_):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.torch = torch
        self.tok = AutoTokenizer.from_pretrained(model, revision=revision)
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        kw = {"revision": revision, "device_map": device_map}
        if load_in_4bit:
            from transformers import BitsAndBytesConfig

            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
        else:
            kw["torch_dtype"] = torch.float16 if dtype in ("half", "float16") else (torch.bfloat16 if dtype == "bfloat16" else torch.float32)
        self.model = AutoModelForCausalLM.from_pretrained(model, **kw).eval()
        self.max_model_len = max_model_len
        self.thinking = enable_thinking
        self.batch_size = batch_size

    def render(self, conv):
        return _apply_template(self.tok, conv, self.thinking)

    def count_tokens(self, conv):
        return len(self.tok(self.render(conv), add_special_tokens=False)["input_ids"])

    def generate(self, convs, max_tokens):
        prompts = [self.render(c) for c in convs]
        lens = [len(self.tok(p, add_special_tokens=False)["input_ids"]) for p in prompts]
        outs = [GenOut("", n, 0, "too_long") for n in lens]
        todo = [i for i, n in enumerate(lens) if n + max_tokens <= self.max_model_len]
        todo.sort(key=lambda i: lens[i])
        for b in range(0, len(todo), self.batch_size):
            idx = todo[b : b + self.batch_size]
            enc = self.tok([prompts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(self.model.device)
            with self.torch.no_grad():
                gen = self.model.generate(**enc, max_new_tokens=max_tokens, do_sample=False, temperature=None, top_p=None,
                                          top_k=None, pad_token_id=self.tok.pad_token_id)
            for row, i in zip(gen, idx):
                new = row[enc["input_ids"].shape[1] :]
                n_out = int((new != self.tok.pad_token_id).sum())
                text = self.tok.decode(new, skip_special_tokens=True)
                outs[i] = GenOut(text, lens[i], n_out, "length" if n_out >= max_tokens else "stop")
        return outs


class MockLLM(BaseLLM):
    """Deterministic stand-in for tests and CPU dry-runs: fn(conv) -> text."""

    def __init__(self, fn: Callable[[list[dict]], str], max_model_len: int = 16384, **_):
        self.fn = fn
        self.max_model_len = max_model_len

    def render(self, conv):
        return "\n".join(f"<{m['role']}>{m['content']}" for m in conv)

    def count_tokens(self, conv):
        return len(self.render(conv)) // 3

    def generate(self, convs, max_tokens):
        out = []
        for c in convs:
            n = self.count_tokens(c)
            out.append(GenOut("", n, 0, "too_long") if n + max_tokens > self.max_model_len else GenOut(self.fn(c), n, 1, "stop"))
        return out


def make_llm(cfg: dict, mock_fn: Callable | None = None) -> BaseLLM:
    """cfg: backend (auto|vllm|hf|mock), model, revision, dtype, max_model_len, gpu_memory_utilization, quantization,
    load_in_4bit, enable_thinking, seed."""
    backend = cfg.get("backend", "auto")
    seed_everything(int(cfg.get("seed", 0)))
    if backend == "mock":
        return MockLLM(mock_fn or (lambda conv: "{\"done\": true}"), **cfg)
    if backend in ("auto", "vllm"):
        try:
            return VLLM(**{k: v for k, v in cfg.items() if k != "backend"})
        except Exception as e:  # not installed, or no kernel for this GPU (e.g. T4): fall back to transformers
            if backend == "vllm":
                raise
            print(f"[llm] vLLM unavailable ({type(e).__name__}: {e}); using transformers", flush=True)
    return HFLLM(**{k: v for k, v in cfg.items() if k != "backend"})

In [ ]:
%%writefile /kaggle/working/lib/lmlib/sft.py
"""Round-2 SFT/LoRA on real train trajectories, with exact resume across Kaggle sessions.

Examples: for every train_fit sample and every step k of label.tool_calls, the agent prompt (same builder as
inference, real tool_results of earlier steps) -> target JSON of call k; then -> {"done": true} unless the last call
is sys.ask_missing_info. Loss on the target tokens only.

Checkpoint (every `ckpt_every_steps` optimizer steps or `ckpt_every_min` minutes): LoRA adapter, optimizer, scheduler,
grad scaler, RNG states, epoch / cursor in the fixed per-epoch permutation, global step, loss log, config hash.
Written to ckpt-XXXXXX.partial, checksummed, COMPLETE marker, then renamed; the latest two are kept. Base weights are
never copied into checkpoints.

Warm start (`init_adapter`): with no checkpoint of its own, the run loads the LoRA weights of the newest complete
checkpoint under that path (another run) and trains with a fresh optimizer and schedule. Its own checkpoints then
resume exactly as usual; the path is not part of the config hash (mount paths change between sessions).
"""
from __future__ import annotations

import argparse
import glob
import json
import math
import os
import random
import shutil
import time
from pathlib import Path

from .agent import Catalog, build_messages
from .asr import gold_texts
from .common import atomic_write_json, config_hash, sha256_file


def _shuffle_apps(r: dict, rng: random.Random, drop_p: float) -> dict:
    """Answer-preserving app augmentation: shuffle the installed apps and drop each app that neither the gold calls nor
    the history use with probability drop_p (the tool list order and size change, the right answer does not)."""
    inst = r["context"].get("installed_apps", {})
    used = {c["name"].split(".")[0] for c in r["label"]["tool_calls"]}
    used |= {str((c.get("arguments") or {}).get("tool_name", "")).split(".")[0] for c in r["label"]["tool_calls"]}
    used |= {c.get("name", "").split(".")[0] for t in r.get("history", []) if t.get("role") == "assistant"
             for c in t.get("tool_calls") or [] if isinstance(c, dict)}
    names = [a for a in inst if a in used or rng.random() >= drop_p]
    rng.shuffle(names)
    return dict(r, context=dict(r["context"], installed_apps={a: inst[a] for a in names}))


def oversample(rows: list[dict], ask: int = 1, calls4: int = 1) -> list[dict]:
    """Repeat the samples the model is weak on: label ending with sys.ask_missing_info `ask` times, 4+ calls `calls4`
    times. Copies get an id suffix (~k), so their augmentation / transcript draw differs from the original."""
    out = []
    for r in rows:
        c = r["label"]["tool_calls"]
        k = ask if c and c[-1]["name"] == "sys.ask_missing_info" else calls4 if len(c) >= 4 else 1
        out += [r] + [dict(r, id=f"{r['id']}~{j}") for j in range(1, k)]
    return out


def build_examples(rows: list[dict], catalog: Catalog, result_chars: int = 1500, texts: dict | None = None,
                   asr_prob: float | None = None, aug: bool = False, drop_p: float = 0.3, seed: int = 0) -> list[dict]:
    """texts: {audio path: transcript} for the prompts (an ASR cache); None or a missing audio -> gold audio_text.
    asr_prob: None -> every sample uses `texts`; p -> each sample (seeded by id) uses the ASR transcripts with
    probability p and the gold ones otherwise, for its final request and history alike.
    aug: shuffle / drop unused installed apps per sample (_shuffle_apps), seeded by id."""
    gold = gold_texts(rows)
    asr = {**gold, **(texts or {})}
    have_asr = texts is not None
    out = []
    for r in rows:
        rng = random.Random(f"{seed}:{r['id']}")
        use_asr = have_asr and (asr_prob is None or rng.random() < asr_prob)
        texts = asr if use_asr else gold
        if aug:
            r = _shuffle_apps(r, rng, drop_p)
        request = texts.get(r["audio"], r["audio_text"])
        calls, results = r["label"]["tool_calls"], r["label"]["tool_results"]
        steps = []
        for k, c in enumerate(calls):
            out.append({"id": f"{r['id']}#{k}", "messages": build_messages(r, request, catalog, texts, list(steps), result_chars),
                        "target": json.dumps({"name": c["name"], "arguments": c.get("arguments") or {}}, ensure_ascii=False)})
            steps.append({"kind": "call", "name": c["name"], "arguments": c.get("arguments") or {}, "result": results[k] if k < len(results) else {}})
        if not calls or calls[-1]["name"] != "sys.ask_missing_info":
            out.append({"id": f"{r['id']}#done", "messages": build_messages(r, request, catalog, texts, list(steps), result_chars),
                        "target": '{"done": true}'})
    return out


def tokenize(ex: dict, tok, max_len: int, thinking: bool = False):
    try:
        prompt = tok.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=True, enable_thinking=thinking)
    except TypeError:
        prompt = tok.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=True)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    t = tok(ex["target"] + (tok.eos_token or ""), add_special_tokens=False)["input_ids"]
    if len(p) + len(t) > max_len:
        return None
    return {"input_ids": p + t, "labels": [-100] * len(p) + t}


def _rng_state():
    import numpy as np
    import torch

    st = {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state()}
    if torch.cuda.is_available():
        st["cuda"] = torch.cuda.get_rng_state_all()
    return st


def _set_rng_state(st):
    import numpy as np
    import torch

    random.setstate(st["python"])
    np.random.set_state(st["numpy"])
    torch.set_rng_state(st["torch"])
    if "cuda" in st and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(st["cuda"])


def latest_checkpoint(dirs) -> Path | None:
    found = []
    for d in dirs:
        for p in glob.glob(str(Path(d) / "**" / "ckpt-*"), recursive=True):
            p = Path(p)
            if p.is_dir() and not p.name.endswith(".partial") and (p / "COMPLETE").exists():
                found.append((int(p.name.split("-")[1]), p))
    return max(found)[1] if found else None


def save_checkpoint(out_dir: Path, model, opt, sched, scaler, state: dict, keep: int = 2) -> Path:
    import torch
    from peft import get_peft_model_state_dict

    final = out_dir / f"ckpt-{state['global_step']:06d}"
    tmp = final.with_name(final.name + ".partial")
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    torch.save(get_peft_model_state_dict(model), tmp / "adapter_state.pt")
    model.peft_config["default"].save_pretrained(str(tmp))
    torch.save({"optimizer": opt.state_dict(), "scheduler": sched.state_dict(), "scaler": scaler.state_dict() if scaler else None,
                "rng": _rng_state(), **state}, tmp / "training_state.pt")
    sums = {p.name: sha256_file(p) for p in sorted(tmp.iterdir())}
    atomic_write_json(tmp / "checksums.json", sums)
    (tmp / "COMPLETE").write_text(state["config_hash"])
    shutil.rmtree(final, ignore_errors=True)
    tmp.rename(final)
    ckpts = sorted(p for p in out_dir.glob("ckpt-*") if p.is_dir() and not p.name.endswith(".partial"))
    for old in ckpts[:-keep]:
        shutil.rmtree(old, ignore_errors=True)
    return final


def _sdpa_without_math_fallback():
    """T4 (sm75) has no flash / cuDNN SDPA, and the mem-efficient kernel rejects GQA (Qwen3-4B: 32 q / 8 kv heads), so
    transformers' enable_gqa=True fell back to the math kernel: 32 x L x L fp16 scores per layer (1.5 GiB at ~5k
    tokens) -> OOM in backward (lm-sft-013). Repeat K/V in transformers instead (mem-efficient kernel), and disable the
    math kernel so any other fallback fails loudly instead of eating the GPU."""
    import torch
    import transformers.integrations.sdpa_attention as sdpa

    sdpa.use_gqa_in_sdpa = lambda *a, **k: False
    torch.backends.cuda.enable_math_sdp(False)


def train(cfg: dict, examples: list[dict] | None = None, model=None, tok=None, log=print) -> dict:
    """cfg keys: model, revision, load_in_4bit, max_len, lora_r, lora_alpha, lora_dropout, target_modules, lr,
    warmup_ratio, epochs, micro_batch (1), grad_accum, max_steps, seed, ckpt_every_steps, ckpt_every_min,
    wallclock_s, out_dir, resume_dirs, gradient_checkpointing, init_adapter."""
    import numpy as np
    import torch
    from peft import LoraConfig, get_peft_model, set_peft_model_state_dict

    out_dir = Path(cfg["out_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)
    seed = int(cfg.get("seed", 0))
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    chash = config_hash(cfg, ignore=("out_dir", "resume_dirs", "wallclock_s", "ckpt_every_min", "max_steps_this_session", "init_adapter"))
    if model is None:
        from transformers import AutoModelForCausalLM, AutoTokenizer

        tok = AutoTokenizer.from_pretrained(cfg["model"], revision=cfg.get("revision"))
        _sdpa_without_math_fallback()
        kw = {"revision": cfg.get("revision"), "device_map": {"": 0}, "attn_implementation": "sdpa"}
        if cfg.get("load_in_4bit"):
            from transformers import BitsAndBytesConfig

            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
        else:
            kw["torch_dtype"] = torch.float16
        model = AutoModelForCausalLM.from_pretrained(cfg["model"], **kw)
        if cfg.get("load_in_4bit"):
            from peft import prepare_model_for_kbit_training

            model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=cfg.get("gradient_checkpointing", True))
        elif cfg.get("gradient_checkpointing", True):
            model.gradient_checkpointing_enable()
            model.enable_input_require_grads()
    lcfg = LoraConfig(r=cfg.get("lora_r", 16), lora_alpha=cfg.get("lora_alpha", 32), lora_dropout=cfg.get("lora_dropout", 0.05),
                      target_modules=cfg.get("target_modules", ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]), task_type="CAUSAL_LM")
    model = get_peft_model(model, lcfg)
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()  # fp32 master weights for LoRA; fp16 autocast for the forward
    data = [x for x in (tokenize(e, tok, int(cfg.get("max_len", 8192))) for e in examples) if x is not None]
    log(f"[sft] {len(data)}/{len(examples)} examples fit max_len")
    accum = int(cfg.get("grad_accum", 16))
    steps_per_epoch = math.ceil(len(data) / accum)
    total = int(cfg.get("max_steps") or steps_per_epoch * int(cfg.get("epochs", 1)))
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=float(cfg.get("lr", 1e-4)), weight_decay=0.0)
    warm = max(1, int(total * float(cfg.get("warmup_ratio", 0.03))))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(total - warm, 1)))
    use_scaler = torch.cuda.is_available() and not cfg.get("load_in_4bit_bf16", False)
    scaler = torch.amp.GradScaler("cuda") if use_scaler else None
    state = {"global_step": 0, "epoch": 0, "cursor": 0, "losses": [], "config_hash": chash}
    ck = latest_checkpoint([out_dir] + list(cfg.get("resume_dirs", [])))
    if ck is not None:
        st = torch.load(ck / "training_state.pt", map_location="cpu", weights_only=False)
        if st["config_hash"] != chash:
            raise RuntimeError(f"checkpoint {ck} was made with another config ({st['config_hash']} != {chash})")
        set_peft_model_state_dict(model, torch.load(ck / "adapter_state.pt", map_location="cpu"))
        opt.load_state_dict(st["optimizer"]); sched.load_state_dict(st["scheduler"])
        if scaler and st.get("scaler"):
            scaler.load_state_dict(st["scaler"])
        _set_rng_state(st["rng"])
        state = {k: st[k] for k in ("global_step", "epoch", "cursor", "losses", "config_hash")}
        log(f"[sft] resumed from {ck} at step {state['global_step']} epoch {state['epoch']} cursor {state['cursor']}")
    elif cfg.get("init_adapter"):
        init = latest_checkpoint([cfg["init_adapter"]])
        if init is None:
            raise FileNotFoundError(f"init_adapter: no complete ckpt-* under {cfg['init_adapter']}")
        set_peft_model_state_dict(model, torch.load(init / "adapter_state.pt", map_location="cpu"))
        log(f"[sft] warm start from {init} (adapter weights only, fresh optimizer / schedule)")
    model.train()
    dev = next(model.parameters()).device
    t0 = last_ck = time.time()
    wall = float(cfg.get("wallclock_s", 3.5 * 3600))
    session_limit = cfg.get("max_steps_this_session")
    session_steps = 0
    stop_reason = "finished"
    while state["global_step"] < total:
        perm = np.random.RandomState(seed + state["epoch"]).permutation(len(data)).tolist()
        while state["cursor"] < len(perm) and state["global_step"] < total:
            idx = perm[state["cursor"] : state["cursor"] + accum]
            tot_tok = sum(sum(l != -100 for l in data[i]["labels"]) for i in idx)
            loss_sum = 0.0
            for i in idx:
                ids = torch.tensor([data[i]["input_ids"]], device=dev)
                n_tgt = sum(l != -100 for l in data[i]["labels"])
                keep = n_tgt + 1  # logits only where a target token is predicted (full-vocab logits of 6k tokens do not fit a T4)
                with torch.autocast("cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                    logits = model(input_ids=ids, logits_to_keep=keep).logits[:, :-1].float()
                tgt = torch.tensor([data[i]["labels"][-n_tgt:]], device=dev)
                loss = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1), ignore_index=-100, reduction="sum") / max(tot_tok, 1)
                (scaler.scale(loss) if scaler else loss).backward()
                loss_sum += float(loss)
            if scaler:
                scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            if scaler:
                scaler.step(opt); scaler.update()
            else:
                opt.step()
            opt.zero_grad(set_to_none=True)
            sched.step()
            state["cursor"] += len(idx)
            state["global_step"] += 1
            session_steps += 1
            state["losses"].append(round(loss_sum, 5))
            if state["global_step"] % 10 == 0:
                log(f"[sft] step {state['global_step']}/{total} loss {loss_sum:.4f} lr {sched.get_last_lr()[0]:.2e} {time.time() - t0:.0f}s")
            due = state["global_step"] % int(cfg.get("ckpt_every_steps", 50)) == 0 or time.time() - last_ck > 60 * float(cfg.get("ckpt_every_min", 20))
            stop = time.time() - t0 > wall or (session_limit and session_steps >= int(session_limit))
            if due or stop:
                save_checkpoint(out_dir, model, opt, sched, scaler, state)
                last_ck = time.time()
            if stop:
                stop_reason = "wallclock" if time.time() - t0 > wall else "session_step_limit"
                break
        if stop_reason != "finished":
            break
        if state["cursor"] >= len(data):
            state["epoch"] += 1
            state["cursor"] = 0
    if stop_reason == "finished":
        save_checkpoint(out_dir, model, opt, sched, scaler, state)
    summary = {"stop_reason": stop_reason, "global_step": state["global_step"], "total_steps": total, "epoch": state["epoch"],
               "examples": len(data), "last_losses": state["losses"][-10:], "session_s": round(time.time() - t0, 1), "config_hash": chash}
    atomic_write_json(out_dir / "train_summary.json", summary)
    summary["model"] = model
    return summary


def resume_selftest(out_dir: str, steps: int = 40) -> dict:
    """Tiny random model on CPU/GPU: `steps` straight vs `steps/2` + resume + `steps/2`; losses must agree."""
    import torch
    from transformers import LlamaConfig, LlamaForCausalLM

    def fresh():
        torch.manual_seed(0)
        m = LlamaForCausalLM(LlamaConfig(vocab_size=128, hidden_size=32, intermediate_size=64, num_hidden_layers=2, num_attention_heads=4, num_key_value_heads=4))
        return m.to("cuda" if torch.cuda.is_available() else "cpu")

    rnd = random.Random(0)
    ex = [{"input_ids": [rnd.randrange(128) for _ in range(24)]} for _ in range(64)]

    class Tok:
        eos_token = ""

        def apply_chat_template(self, m, **k):
            return m[0]["content"]

        def __call__(self, s, add_special_tokens=False):
            return {"input_ids": [int(x) for x in s.split()] if s else []}

    tok = Tok()
    examples = [{"messages": [{"role": "user", "content": " ".join(map(str, e["input_ids"][:12]))}], "target": " ".join(map(str, e["input_ids"][12:]))} for e in ex]
    base = {"max_len": 64, "lora_r": 4, "lora_alpha": 8, "lora_dropout": 0.0, "target_modules": ["q_proj", "v_proj"], "lr": 1e-3,
            "grad_accum": 2, "max_steps": steps, "seed": 0, "ckpt_every_steps": 1000, "gradient_checkpointing": False}
    a = train({**base, "out_dir": str(Path(out_dir) / "straight")}, examples, model=fresh(), tok=tok, log=lambda *_: None)
    train({**base, "out_dir": str(Path(out_dir) / "resumed"), "max_steps_this_session": steps // 2}, examples, model=fresh(), tok=tok, log=lambda *_: None)
    b = train({**base, "out_dir": str(Path(out_dir) / "resumed")}, examples, model=fresh(), tok=tok, log=lambda *_: None)
    la = json.loads(json.dumps(a["last_losses"]))
    lb = json.loads(json.dumps(b["last_losses"]))
    diff = max(abs(x - y) for x, y in zip(la, lb))
    return {"straight_last": la, "resumed_last": lb, "max_abs_diff": diff, "ok": diff < 1e-3 and a["global_step"] == b["global_step"] == steps}


def merge_adapter(base_model: str, ckpt: Path, out: Path, revision: str | None = None) -> Path:
    """Base fp16 + LoRA -> merged weights the agent notebook can load as `model`."""
    import torch
    from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(base_model, revision=revision)
    m = AutoModelForCausalLM.from_pretrained(base_model, revision=revision, torch_dtype=torch.float16)
    m = get_peft_model(m, LoraConfig.from_pretrained(str(ckpt)))
    set_peft_model_state_dict(m, torch.load(ckpt / "adapter_state.pt", map_location="cpu"))
    m = m.merge_and_unload()
    m.save_pretrained(str(out), safe_serialization=True)
    tok.save_pretrained(str(out))
    return out


def merge_latest(base_model: str, ckpt_dirs, out: Path, revision: str | None = None) -> Path:
    """Merge the newest complete checkpoint found under `ckpt_dirs` (dir or list of dirs) into `out`."""
    ck = latest_checkpoint([ckpt_dirs] if isinstance(ckpt_dirs, (str, Path)) else ckpt_dirs)
    if ck is None:
        raise FileNotFoundError(f"no complete ckpt-* under {ckpt_dirs}")
    print(f"[sft] merging {ck} -> {out}", flush=True)
    return merge_adapter(base_model, ck, Path(out), revision)


def main(argv=None):
    """python -m lmlib.sft --config cfg.json  (cfg = train() keys + examples_path)"""
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    a = ap.parse_args(argv)
    cfg = json.loads(Path(a.config).read_text(encoding="utf-8"))
    with open(cfg["examples_path"], encoding="utf-8") as f:
        examples = [json.loads(l) for l in f if l.strip()]
    summary = train(cfg, examples, log=lambda *x: print(*x, flush=True))
    summary.pop("model", None)
    print(json.dumps(summary, ensure_ascii=False, indent=1))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/lib/lmlib/provenance.py
"""Content fingerprints and strict shard collection; execution metadata is separate."""
from __future__ import annotations
import hashlib
import inspect
import json
import shutil
from pathlib import Path
from .common import atomic_write_json, read_jsonl_tolerant, sha256_file

RUNTIME_KEYS = {'execution_id','parent_execution_id','account_alias','kaggle_version','timestamp','out_dir','input_roots','split_dir','cache_dir','asr_cache_dirs','resume_from_dirs','resume','wallclock_s','session_wallclock_s','session_started_epoch','session_deadline_epoch','export_reserve_s','batch_estimate_s','preflight_dirs','control_dir','candidate_dir','lib_root','llm_python','executor_python','num_gpus','shard_index','notes','parent_run','source_execution','source_version','asr_expectation','code_sha256','verify_reuse','asr_reused_from'}

def digest(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=False,allow_nan=False,default=str,separators=(',',':')).encode()).hexdigest()

def source_hash(obj):
    return hashlib.sha256(inspect.getsource(obj).encode()).hexdigest()

def package_hash(package_dir):
    return digest({p.name:sha256_file(p) for p in sorted(Path(package_dir).glob('*.py')) if p.name not in {'regression.py'}})

def tree_hash(directory):
    root=Path(directory).resolve()
    records={}
    for p in sorted(root.rglob('*')):
        if p.is_file():
            if not p.resolve().is_relative_to(root):raise ValueError(f'source symlink escapes root: {p}')
            records[str(p.relative_to(root))]=sha256_file(p)
    return digest(records)

def semantic_hash(cfg):
    semantic={k:v for k,v in cfg.items() if k not in RUNTIME_KEYS}
    if cfg.get('experiment_id') and cfg.get('experiment') != cfg['experiment_id']:
        raise ValueError('experiment and experiment_id must agree')
    # Interpreter and source identities are content-addressed in preflight/manifest, never by mount path.
    semantic['code_sha256']=cfg.get('code_sha256') or package_hash(Path(__file__).parent)
    if cfg.get('asr_expectation'):
        semantic['asr_content_key']=cfg['asr_expectation']['content_key']
    return digest(semantic)

def freeze_json(path,value):
    p=Path(path)
    if p.exists() and json.loads(p.read_text()) != value:raise ValueError(f'frozen artifact differs: {p}')
    atomic_write_json(p,value)
    return value

def collect_shards(out_dir,pattern,expected_hash=None,allowed_ids=None):
    records={};expected=set(allowed_ids) if allowed_ids is not None else None
    for p in sorted(Path(out_dir).glob(pattern)):
        rows,_=read_jsonl_tolerant(p)
        header=rows[0].get('_header') if rows else None
        if not header or (expected_hash is not None and header.get('run_hash')!=expected_hash):
            raise ValueError(f'shard header mismatch: {p}')
        if expected_hash is None:expected_hash=header.get('run_hash')
        for rec in rows[1:]:
            qid=rec.get('id')
            if expected is not None and qid not in expected:raise ValueError(f'unknown ID {qid} in {p}')
            previous=records.get(qid)
            if previous and previous.get('status')=='completed':
                if rec.get('status')!='completed' or digest({k:v for k,v in previous.items() if k not in {'execution_id'}})!=digest({k:v for k,v in rec.items() if k not in {'execution_id'}}):
                    raise ValueError(f'conflicting completed ID: {qid}')
            records[qid]=rec
    return records

def restore_outputs(dirs,destination):
    """Copy explicitly attached checkpoint roots, never scan all inputs or overwrite another lineage."""
    dest=Path(destination);dest.mkdir(parents=True,exist_ok=True)
    for directory in dirs:
        src=Path(directory)
        if not src.is_dir():raise FileNotFoundError(src)
        for p in src.rglob('*'):
            if not p.is_file() or p.suffix not in {'.json','.jsonl'}:continue
            target=dest/p.relative_to(src)
            if target.exists():
                if target.read_bytes()!=p.read_bytes():
                    if p.name in {'environment.json','wheels_manifest.json'}:continue
                    raise ValueError(f'resume destination already differs: {target}')
            else:
                target.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(p,target)


In [ ]:
%%writefile /kaggle/working/lib/lmlib/qwen35smoke.py
"""lm-sft-038 smoke: can the lm-sft-021 LoRA recipe train Qwen/Qwen3.5-4B on a Kaggle T4, merge it and serve it with vLLM?

    python -m lmlib.qwen35smoke probe --config cfg.json   # load, base losses, LoRA fwd+bwd on long examples -> report
    python -m lmlib.qwen35smoke train --config cfg.json   # lmlib.sft.train (the real loop) + peak VRAM -> report
    python -m lmlib.qwen35smoke merge --config cfg.json   # CPU: LoRA merged, written into a copy of the original checkpoint
    python -m lmlib.qwen35smoke vllm  --config cfg.json   # (vLLM venv) load the merged checkpoint, generate dev steps

Switches in cfg: block_kernels hides fla / causal_conv1d, so transformers binds its torch gated-delta path (the
Triton kernels of fla are not proven on sm75); allow_math_sdp keeps the SDPA math kernel that lmlib.sft turns off.
Every command writes cfg["report_path"] even when it fails, so the notebook can compare paths instead of dying.
"""
from __future__ import annotations

import argparse
import json
import shutil
import sys
import time
import traceback
from pathlib import Path


def block_kernels() -> None:
    for m in ("fla", "causal_conv1d"):
        sys.modules[m] = None  # import_module -> ImportError, transformers keeps its torch implementation


def apply_switches(cfg: dict) -> None:
    if cfg.get("block_kernels"):
        block_kernels()
    if cfg.get("allow_math_sdp"):
        from . import sft

        def _gqa_repeat_only():
            import transformers.integrations.sdpa_attention as sdpa

            sdpa.use_gqa_in_sdpa = lambda *a, **k: False

        sft._sdpa_without_math_fallback = _gqa_repeat_only


def kernel_impls() -> dict:
    """module.name of the functions transformers bound for the Qwen3.5 gated-delta / conv ops."""
    try:
        import transformers.models.qwen3_5.modeling_qwen3_5 as mq
    except ImportError:
        return {}
    out = {}
    for n in ("torch_chunk_gated_delta_rule", "torch_recurrent_gated_delta_rule", "causal_conv1d_fn"):
        fn = getattr(mq, n, None)
        code = getattr(fn, "__code__", None)
        cells = dict(zip(code.co_freevars, (c.cell_contents for c in fn.__closure__ or ()))) if code else {}
        impl = cells.get("implementation", fn)
        out[n] = f"{getattr(impl, '__module__', '?')}.{getattr(impl, '__name__', '?')}"
    return out


def _loss(model, d: dict, device):
    """Same loss as lmlib.sft.train: logits only where a target token is predicted, mean over target tokens."""
    import torch

    n_tgt = sum(l != -100 for l in d["labels"])
    ids = torch.tensor([d["input_ids"]], device=device)
    with torch.autocast("cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
        logits = model(input_ids=ids, logits_to_keep=n_tgt + 1).logits[:, :-1].float()
    tgt = torch.tensor([d["labels"][-n_tgt:]], device=device)
    return torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1), ignore_index=-100)


def _peak_gb() -> float:
    import torch

    return round(torch.cuda.max_memory_allocated() / 2**30, 2) if torch.cuda.is_available() else 0.0


def _err(e: BaseException) -> str:
    return f"{type(e).__name__}: {str(e)[:600]}"


def _percentile(xs: list[int], q: float) -> int:
    return xs[min(len(xs) - 1, int(q * len(xs)))] if xs else 0


def probe(cfg: dict) -> dict:
    import torch
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer

    from . import sft

    device = cfg.get("device", "cuda:0")
    rep = {"model": cfg["model"], "block_kernels": bool(cfg.get("block_kernels")), "allow_math_sdp": bool(cfg.get("allow_math_sdp"))}
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(cfg["model"], revision=cfg.get("revision"))
    sft._sdpa_without_math_fallback()
    model, info = AutoModelForCausalLM.from_pretrained(cfg["model"], revision=cfg.get("revision"), device_map={"": device},
                                                       attn_implementation="sdpa", dtype=getattr(torch, cfg.get("dtype", "float16")),
                                                       output_loading_info=True)
    rep.update(load_s=round(time.time() - t0, 1), cls=type(model).__name__, params=sum(p.numel() for p in model.parameters()),
               missing_keys=len(info.get("missing_keys", [])), missing_head=sorted(info.get("missing_keys", []))[:5],
               kernels=kernel_impls(), vram_after_load_gb=_peak_gb())
    examples = [json.loads(l) for l in open(cfg["examples_path"], encoding="utf-8")]
    data = [sft.tokenize(e, tok, 10**9) for e in examples]
    lens = sorted(len(d["input_ids"]) for d in data)
    max_len = int(cfg.get("max_len", 8192))
    rep["tokens"] = {"n": len(lens), "p50": _percentile(lens, 0.5), "p95": _percentile(lens, 0.95), "max": lens[-1],
                     "over_max_len": sum(n > max_len for n in lens), "total": sum(lens)}
    model.eval()
    try:  # base model, no LoRA: finite and sane (fp16 overflow / unloaded weights show up here)
        with torch.no_grad():
            rep["base_losses"] = [round(float(_loss(model, data[i], device)), 4) for i in cfg["probe_ids"]]
    except Exception as e:
        rep["base_error"] = _err(e)
        return rep
    if cfg.get("gradient_checkpointing", True):
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(r=cfg.get("lora_r", 16), lora_alpha=cfg.get("lora_alpha", 32), lora_dropout=cfg.get("lora_dropout", 0.05),
                                             target_modules=cfg["target_modules"], task_type="CAUSAL_LM"))
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    rep["lora_params"] = sum(p.numel() for p in model.parameters() if p.requires_grad)
    rep["lora_modules"] = sorted({n.split(".lora_A")[0].split(".")[-1] for n, p in model.named_parameters() if p.requires_grad and ".lora_A" in n})
    model.train()
    fit = sorted((i for i, d in enumerate(data) if len(d["input_ids"]) <= max_len), key=lambda i: len(data[i]["input_ids"]))
    rep["fwd_bwd"] = {}
    for tag, i in (("p95", fit[min(len(fit) - 1, int(0.95 * len(fit)))]), ("max", fit[-1])):
        r = {"tokens": len(data[i]["input_ids"])}
        try:
            times = []
            for _ in range(2):  # the first call also compiles / autotunes Triton kernels
                if torch.cuda.is_available():
                    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
                t = time.time()
                loss = _loss(model, data[i], device)
                loss.backward()
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                times.append(round(time.time() - t, 2))
                grads = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
                r["grads_finite"] = bool(grads) and all(torch.isfinite(g).all().item() for g in grads)
                model.zero_grad(set_to_none=True)
            r.update(loss=round(float(loss), 4), s=times, peak_gb=_peak_gb())
        except Exception as e:
            r["error"] = _err(e)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        rep["fwd_bwd"][tag] = r
    return rep


def train_cmd(cfg: dict) -> dict:
    import torch

    from .sft import train

    examples = [json.loads(l) for l in open(cfg["examples_path"], encoding="utf-8")]
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    s = train(cfg, examples, log=lambda *x: print(*x, flush=True))
    s.pop("model", None)
    steps = max(int(s["global_step"]), 1)
    losses = json.loads((Path(cfg["out_dir"]) / "train_summary.json").read_text())["last_losses"]
    return dict(s, s_per_step=round(s["session_s"] / steps, 1), peak_gb=_peak_gb(), kernels=kernel_impls(),
                losses_finite=all(l == l and abs(l) != float("inf") for l in losses))


def merge_for_vllm(base: str, ckpt_dirs: list, out: Path, revision: str | None = None) -> dict:
    """LoRA merged into the text-only model, then written over a copy of the original Qwen3_5ForConditionalGeneration
    checkpoint (configs, tokenizer, vision and mtp tensors untouched) so vLLM loads it exactly like lm-B-011."""
    import torch
    from huggingface_hub import snapshot_download
    from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
    from safetensors.torch import load_file, save_file
    from transformers import AutoModelForCausalLM

    from .sft import latest_checkpoint

    ck = latest_checkpoint(ckpt_dirs)
    if ck is None:
        raise FileNotFoundError(f"no complete ckpt-* under {ckpt_dirs}")
    src = Path(base) if Path(base).is_dir() else Path(snapshot_download(base, revision=revision))
    m = AutoModelForCausalLM.from_pretrained(str(src), dtype=torch.float16)
    m = get_peft_model(m, LoraConfig.from_pretrained(str(ck)))
    set_peft_model_state_dict(m, torch.load(ck / "adapter_state.pt", map_location="cpu"))
    m = m.merge_and_unload()
    merged = {"model.language_model." + k[len("model."):]: v for k, v in m.state_dict().items() if k.startswith("model.")}
    del m
    out.mkdir(parents=True, exist_ok=True)
    for f in src.iterdir():
        if f.is_file() and not f.name.endswith(".safetensors"):
            shutil.copy(f, out / f.name)
    used, changed = set(), 0
    for shard in sorted(src.glob("*.safetensors")):
        t = load_file(str(shard))
        for k in t:
            if k in merged:
                new = merged[k].to(torch.float16).contiguous()
                changed += int(not torch.equal(new, t[k].to(torch.float16)))
                t[k] = new
                used.add(k)
        save_file(t, str(out / shard.name), metadata={"format": "pt"})
        del t
    missing = sorted(set(merged) - used)
    return {"ckpt": str(ck), "out": str(out), "replaced": len(used), "changed": changed, "n_missing": len(missing), "missing_head": missing[:5]}


def vllm_check(cfg: dict) -> dict:
    from .agent import parse_action
    from .llm import VLLM

    ex = [json.loads(l) for l in open(cfg["examples_path"], encoding="utf-8")]
    t = time.time()
    llm = VLLM(**cfg["llm"])
    load_s = round(time.time() - t, 1)
    t = time.time()
    outs = llm.generate([e["messages"] for e in ex], int(cfg.get("max_tokens", 256)))
    gen_s = time.time() - t
    valid = name = exact = 0
    samples = []
    for e, o in zip(ex, outs):
        kind, n, args = parse_action(o.text)
        gold = parse_action(e["target"])
        valid += kind != "invalid"
        name += kind == gold[0] and n == gold[1]
        exact += (kind, n, args) == gold
        if len(samples) < 4:
            samples.append({"target": e["target"][:200], "output": o.text[:200], "finish": o.finish})
    k = max(len(ex), 1)
    return {"n": len(ex), "load_s": load_s, "s_per_prompt": round(gen_s / k, 2), "valid_json": round(valid / k, 3),
            "name_match": round(name / k, 3), "exact_match": round(exact / k, 3), "finish": sorted({o.finish for o in outs}),
            "samples": samples}


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["probe", "train", "merge", "vllm"])
    ap.add_argument("--config", required=True)
    a = ap.parse_args(argv)
    cfg = json.loads(Path(a.config).read_text(encoding="utf-8"))
    apply_switches(cfg)
    t = time.time()
    try:
        if a.cmd == "probe":
            rep = probe(cfg)
        elif a.cmd == "train":
            rep = train_cmd(cfg)
        elif a.cmd == "merge":
            rep = merge_for_vllm(cfg["model"], cfg["ckpt_dirs"], Path(cfg["merged_dir"]), cfg.get("revision"))
        else:
            rep = vllm_check(cfg)
        rep["ok"] = not any(k in rep for k in ("error", "base_error"))
    except Exception as e:
        traceback.print_exc()
        rep = {"ok": False, "error": _err(e)}
    rep.update(cmd=a.cmd, wall_s=round(time.time() - t, 1))
    Path(cfg["report_path"]).write_text(json.dumps(rep, ensure_ascii=False, indent=1), encoding="utf-8")
    print(json.dumps(rep, ensure_ascii=False)[:3000], flush=True)


if __name__ == "__main__":
    main()


In [ ]:
# Thư viện cài thêm: ưu tiên wheel từ output `build-wheels` (đã gắn sẵn), lỗi thì cài từ mạng
WHEEL_DIRS = sorted({str(Path(p).parent) for p in glob.glob("/kaggle/input/**/wheels/*.whl", recursive=True)})
print("wheel dirs:", WHEEL_DIRS or "none -> install from internet")

# Model tải từ huggingface.co: tắt Internet thì dừng ngay thay vì chết giữa chừng (lm-B-020 v2, lm-A-030 lần 1)
import socket
try:
    socket.create_connection(("huggingface.co", 443), timeout=15).close()
except OSError as e:
    raise RuntimeError(f"không kết nối được huggingface.co ({e}): bật Settings → Internet On rồi Save & Run All lại") from e

import torch
N_GPUS = torch.cuda.device_count()
print("GPUs:", N_GPUS, [torch.cuda.get_device_name(i) for i in range(N_GPUS)])
if N_GPUS < 2:
    raise RuntimeError("cần GPU T4 x2 (Settings → Accelerator → GPU T4 x2)")

def pip_install(pkgs: str, extra: str = "") -> int:
    if WHEEL_DIRS:
        links = " ".join(f"--find-links {d}" for d in WHEEL_DIRS)
        if sh(f"{sys.executable} -m pip install -q --no-index {links} {extra} {pkgs}") == 0:
            print("installed from wheels:", pkgs)
            return 0
        print("wheel install failed -> internet:", pkgs)
    return sh(f"{sys.executable} -m pip install -q {extra} {pkgs}")

# Gói nhẹ, cài thẳng môi trường gốc nhưng ghim torch / triton / transformers đang có (không để pip đổi stack CUDA)
from importlib.metadata import PackageNotFoundError, version
PINS = {}
for pkg in ("torch", "triton", "transformers", "numpy"):
    try:
        PINS[pkg] = version(pkg)
    except PackageNotFoundError:
        pass
CONSTRAINTS = WORK / "constraints.txt"
CONSTRAINTS.write_text("".join(f"{k}=={v}\n" for k, v in PINS.items()))
print("pinned:", PINS)

# PEFT >=0.20 kiểm torchao ngay khi gắn LoRA; torchao 0.10 của image không tương thích (lm-sft-013)
if pip_install("peft", f"-c {CONSTRAINTS}") != 0:
    raise RuntimeError("PEFT installation failed")
from packaging.version import Version
try:
    torchao_version = version("torchao")
except PackageNotFoundError:
    torchao_version = None
if torchao_version is not None and Version(torchao_version) < Version("0.16.0"):
    print("removing unused incompatible torchao:", torchao_version)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)

# Kernel Triton cho Gated DeltaNet (transformers tự dùng nếu import được `fla`). Không cài được → chỉ đo đường torch.
FLA_OK = sh(f"{sys.executable} -m pip install -q -c {CONSTRAINTS} flash-linear-attention") == 0 and \
         sh(f'{sys.executable} -c "import fla, triton; print(\'fla\', getattr(fla, \'__version__\', \'?\'), \'triton\', triton.__version__)"') == 0
print("fla importable:", FLA_OK)

# vLLM ở venv riêng (không bao giờ cài vào môi trường gốc: thay torch → hỏng ASR/transformers, lm-B-002)
VENV = Path("/tmp/venv_vllm")
pip_install("uv")
sh(f"{sys.executable} -m uv venv -q {VENV} --python {sys.executable}")
VPY = VENV / "bin" / "python"
rc = 1
if WHEEL_DIRS:
    links = " ".join(f"--find-links {d}" for d in WHEEL_DIRS)
    rc = sh(f"{sys.executable} -m uv pip install -q --python {VPY} --no-index {links} {CONFIG['vllm_spec']}")
if rc != 0:
    rc = sh(f"{sys.executable} -m uv pip install -q --python {VPY} {CONFIG['vllm_spec']}")
VLLM_OK = rc == 0 and sh(f'{VPY} -c "import vllm, torch; assert torch.cuda.is_available(); print(\'vllm\', vllm.__version__, \'torch\', torch.__version__)"') == 0
print("vLLM venv ok:", VLLM_OK)


In [ ]:
# Dataset: bản đã gắn sẵn trong /kaggle/input; nếu thiếu thì tự tải bằng kagglehub
# (https://www.kaggle.com/datasets/thanhkhil/zaloai-lifemate-dataset)
DATASET = "thanhkhil/zaloai-lifemate-dataset"
if not glob.glob(f"/kaggle/input/**/{CONFIG['split']}/{CONFIG['split']}.jsonl", recursive=True):
    pip_install("kagglehub")
    import kagglehub
    DATA_PATH = kagglehub.dataset_download(DATASET)
    print("dataset downloaded to", DATA_PATH)
    CONFIG["input_roots"] = [str(DATA_PATH)] + CONFIG["input_roots"]
else:
    print("dataset found under /kaggle/input")


## Example SFT: train_fit (smoke train) và dev (kiểm vLLM), transcript gold

In [ ]:
import random
from transformers import AutoTokenizer
from lmlib.agent import Catalog
from lmlib.common import atomic_write_json
from lmlib.data import load_rows, make_manifest, resolve_split_dir
from lmlib.sft import build_examples, tokenize

SD = resolve_split_dir(CONFIG)
rows = load_rows(SD)
man = make_manifest(rows, seed=CONFIG["manifest_seed"])
by_id = {r["id"]: r for r in rows}
catalog = Catalog(json.loads((SD / f"{SD.name}_tools_catalog.json").read_text(encoding="utf-8")))
fit = [by_id[i] for i in man["ids"]["train_fit"]]
dev = [by_id[i] for i in man["ids"]["dev_seen"] + man["ids"]["dev_unseen"]]
print(man["counts"])

# texts=None -> transcript gold (smoke không cần ASR: độ dài prompt gần như như nhau)
ex_fit = build_examples(fit, catalog, CONFIG["result_chars"], texts=None, seed=CONFIG["sft"]["seed"], **CONFIG["aug"])
ex_all = build_examples(rows, catalog, CONFIG["result_chars"], texts=None, seed=CONFIG["sft"]["seed"], **CONFIG["aug"])
ex_dev = build_examples(dev, catalog, CONFIG["result_chars"], texts=None, seed=0)
print("examples: train_fit", len(ex_fit), "| toàn bộ train", len(ex_all), "| dev", len(ex_dev))

TOK = {k: AutoTokenizer.from_pretrained(CONFIG[k]["model"], revision=CONFIG[k]["revision"]) for k in ("candidate", "baseline")}
LENS = {k: [len(tokenize(e, t, 10**9)["input_ids"]) for e in ex_fit] for k, t in TOK.items()}
max_len = CONFIG["sft"]["max_len"]
TOKSTAT = {}
for k, ls in LENS.items():
    s = sorted(ls)
    TOKSTAT[k] = {"mean": round(sum(s) / len(s)), "p50": s[len(s) // 2], "p95": s[int(0.95 * len(s))], "max": s[-1],
                  "over_max_len": sum(n > max_len for n in s)}
print("token / example train_fit:", TOKSTAT)

# 192 example ngẫu nhiên vừa max_len với cả hai tokenizer: hai model train đúng cùng dữ liệu, cùng thứ tự
both = [i for i in range(len(ex_fit)) if max(LENS["candidate"][i], LENS["baseline"][i]) <= max_len]
n_smoke = CONFIG["smoke_steps"] * CONFIG["sft"]["grad_accum"]
SMOKE_IDX = sorted(random.Random(2026).sample(both, n_smoke))
SMOKE_MEAN = {k: round(sum(LENS[k][i] for i in SMOKE_IDX) / n_smoke) for k in LENS}
print("smoke subset mean tokens:", SMOKE_MEAN)

def write_jsonl(path, items):
    Path(path).write_text("".join(json.dumps(e, ensure_ascii=False) + "\n" for e in items), encoding="utf-8")

write_jsonl(OUT / "fit_examples.jsonl", ex_fit)
write_jsonl(OUT / "smoke_train.jsonl", [ex_fit[i] for i in SMOKE_IDX])
# kiểm vLLM: bước bất kỳ của mẫu dev (không aug), cố định theo seed
write_jsonl(OUT / "vllm_examples.jsonl", random.Random(7).sample(ex_dev, CONFIG["vllm_n"]))
# probe số học: 3 example độ dài trung vị theo tokenizer Qwen3.5
order = sorted(range(len(ex_fit)), key=lambda i: LENS["candidate"][i])
PROBE_IDS = order[len(order) // 2 - 1: len(order) // 2 + 2]
atomic_write_json(OUT / "data_summary.json", {"counts": man["counts"], "n_fit_examples": len(ex_fit), "n_all_examples": len(ex_all),
                  "tokens": TOKSTAT, "smoke_mean_tokens": SMOKE_MEAN, "probe_ids": PROBE_IDS})


## Chạy: GPU0 Qwen3.5-4B, GPU1 Qwen3-4B (song song)

In [ ]:
def worker(cmd, cfg, gpu, name, python=sys.executable):
    """One lmlib.qwen35smoke command in its own process on one GPU; returns its report (always written)."""
    d = OUT / name; d.mkdir(parents=True, exist_ok=True)
    cfg = dict(cfg, report_path=str(d / f"{cmd}.json"))
    cp = d / f"{cmd}_config.json"; atomic_write_json(cp, cfg)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
               TOKENIZERS_PARALLELISM="false", PYTHONPATH=str(LIB) + os.pathsep + os.environ.get("PYTHONPATH", ""))
    t = time.time()
    with open(d / f"{cmd}.log", "w", encoding="utf-8") as log:
        code = subprocess.run([python, "-m", "lmlib.qwen35smoke", cmd, "--config", str(cp)], env=env, stdout=log, stderr=subprocess.STDOUT).returncode
    rp = Path(cfg["report_path"])
    rep = json.loads(rp.read_text()) if rp.exists() else {"ok": False, "error": f"no report (exit {code})"}
    rep["exit"] = code
    print(f"[{name}] {cmd} exit {code} ok={rep.get('ok')} {(time.time() - t) / 60:.1f} min", flush=True)
    if not rep.get("ok"):
        print((d / f"{cmd}.log").read_text(encoding="utf-8")[-3000:], flush=True)
    return rep

def base_cfg(which):
    m = CONFIG[which]
    return dict(CONFIG["sft"], model=m["model"], revision=m["revision"], target_modules=CONFIG["target_modules"][which],
                device="cuda:0", probe_ids=PROBE_IDS, examples_path=str(OUT / "fit_examples.jsonl"))

def train_cfg(which, name, switches):
    return dict(base_cfg(which), **switches, examples_path=str(OUT / "smoke_train.jsonl"), max_steps=CONFIG["smoke_steps"],
                out_dir=str(OUT / name / "sft"), resume_dirs=[])

def no_kernel(rep):  # SDPA từ chối mọi kernel khi tắt math (head_dim 256 trên sm75?)
    text = json.dumps(rep, ensure_ascii=False)
    return "No available kernel" in text

def probe_ok(rep):
    fb = rep.get("fwd_bwd", {})
    return bool(rep.get("ok")) and all(fb.get(t, {}).get("grads_finite") for t in ("p95", "max"))

R = {"candidate": {}, "baseline": {}}

def candidate_chain():
    r = R["candidate"]
    sw = {"block_kernels": True}
    r["probe_torch"] = worker("probe", dict(base_cfg("candidate"), **sw), 0, "q35_torch")
    if no_kernel(r["probe_torch"]):
        sw["allow_math_sdp"] = True
        r["probe_torch"] = worker("probe", dict(base_cfg("candidate"), **sw), 0, "q35_torch_math")
    r["sdp_switch"] = {k: v for k, v in sw.items() if k == "allow_math_sdp"}
    if FLA_OK:
        r["probe_fla"] = worker("probe", dict(base_cfg("candidate"), **r["sdp_switch"]), 0, "q35_fla")
    t, f = r["probe_torch"], r.get("probe_fla", {})
    diff = None
    if t.get("base_losses") and f.get("base_losses"):
        diff = max(abs(a - b) for a, b in zip(t["base_losses"], f["base_losses"]))
    r["fla_loss_diff"] = diff
    fla_fast = probe_ok(f) and diff is not None and diff <= CONFIG["fla_max_loss_diff"] and \
               (not probe_ok(t) or f["fwd_bwd"]["max"]["s"][-1] < t["fwd_bwd"]["max"]["s"][-1])
    if fla_fast:
        r["path"], switches = "fla", dict(r["sdp_switch"])
    elif probe_ok(t):
        r["path"], switches = "torch", dict(r["sdp_switch"], block_kernels=True)
    else:
        r["path"] = None
        return
    r["train"] = worker("train", train_cfg("candidate", "q35_train", switches), 0, "q35_train")
    if not r["train"].get("ok"):
        return
    r["merge"] = worker("merge", dict(base_cfg("candidate"), ckpt_dirs=[str(OUT / "q35_train" / "sft")],
                                      merged_dir="/tmp/q35_merged"), 0, "q35_merge")
    if r["merge"].get("ok") and VLLM_OK:
        r["vllm"] = worker("vllm", {"examples_path": str(OUT / "vllm_examples.jsonl"), "max_tokens": 256,
                                    "llm": dict(CONFIG["vllm"], model="/tmp/q35_merged")}, 0, "q35_vllm", python=str(VPY))

def baseline_chain():
    r = R["baseline"]
    r["probe"] = worker("probe", base_cfg("baseline"), 1, "q3_probe")
    r["train"] = worker("train", train_cfg("baseline", "q3_train", {}), 1, "q3_train")

T0 = time.time()
threads = [threading.Thread(target=candidate_chain), threading.Thread(target=baseline_chain)]
for th in threads: th.start()
for th in threads: th.join()
print(f"xong sau {(time.time() - T0) / 60:.1f} phút")
atomic_write_json(OUT / "reports.json", R)


## Kết quả và quyết định

In [ ]:
import math
ds = json.loads((OUT / "data_summary.json").read_text())
c, b = R["candidate"], R["baseline"]

def est_hours(train_rep, which, n_examples):
    """2 epoch, accum 16; s/step của smoke chỉnh theo độ dài trung bình toàn train_fit / tập smoke."""
    if not train_rep or not train_rep.get("ok"):
        return None
    scale = ds["tokens"][which]["mean"] / ds["smoke_mean_tokens"][which]
    return round(train_rep["s_per_step"] * scale * 2 * math.ceil(n_examples / CONFIG["sft"]["grad_accum"]) / 3600, 1)

def fb(rep, tag, key):
    return (rep or {}).get("fwd_bwd", {}).get(tag, {}).get(key)

probe_c = c.get("probe_fla") if c.get("path") == "fla" else c.get("probe_torch")
rows_tbl = [
    ("kernel Gated DeltaNet đã dùng", c.get("path"), "—"),
    ("|loss fla − torch| (3 example)", c.get("fla_loss_diff"), "—"),
    ("SDPA cần bật math kernel", bool(c.get("sdp_switch", {}).get("allow_math_sdp")), False),
    ("tham số model", (probe_c or {}).get("params"), b.get("probe", {}).get("params")),
    ("missing keys khi load", (probe_c or {}).get("missing_keys"), b.get("probe", {}).get("missing_keys")),
    ("token TB / example train_fit", ds["tokens"]["candidate"]["mean"], ds["tokens"]["baseline"]["mean"]),
    ("example > max_len 8192", ds["tokens"]["candidate"]["over_max_len"], ds["tokens"]["baseline"]["over_max_len"]),
    ("loss base (3 example)", (probe_c or {}).get("base_losses"), b.get("probe", {}).get("base_losses")),
    ("fwd+bwd example dài nhất: s", fb(probe_c, "max", "s"), fb(b.get("probe"), "max", "s")),
    ("fwd+bwd example dài nhất: peak GB", fb(probe_c, "max", "peak_gb"), fb(b.get("probe"), "max", "peak_gb")),
    ("train 12 step: s/step", c.get("train", {}).get("s_per_step"), b.get("train", {}).get("s_per_step")),
    ("train: peak VRAM GB", c.get("train", {}).get("peak_gb"), b.get("train", {}).get("peak_gb")),
    ("train: loss 10 step cuối", c.get("train", {}).get("last_losses"), b.get("train", {}).get("last_losses")),
    ("ước tính 2 epoch train_fit (giờ, 1 GPU)", est_hours(c.get("train"), "candidate", ds["n_fit_examples"]), est_hours(b.get("train"), "baseline", ds["n_fit_examples"])),
    ("ước tính 2 epoch toàn train (giờ, 1 GPU)", est_hours(c.get("train"), "candidate", ds["n_all_examples"]), est_hours(b.get("train"), "baseline", ds["n_all_examples"])),
    ("merge: tensor thay / thay đổi / thiếu", (lambda m: m and f"{m.get('replaced')}/{m.get('changed')}/{m.get('n_missing')}")(c.get("merge")), "—"),
    ("vLLM: load s / s mỗi prompt", (lambda v: v and f"{v.get('load_s')} / {v.get('s_per_prompt')}")(c.get("vllm")), "—"),
    ("vLLM: JSON hợp lệ / đúng tên / khớp hẳn", (lambda v: v and f"{v.get('valid_json')} / {v.get('name_match')} / {v.get('exact_match')}")(c.get("vllm")), "—"),
]
print(f"{'':45s} {'Qwen3.5-4B':>32s} {'Qwen3-4B':>32s}")
for name, x, y in rows_tbl:
    print(f"{name:45s} {str(x):>32s} {str(y):>32s}")

reasons = []
if not c.get("path"):
    reasons.append("không train được ở cả hai đường kernel (xem log q35_*/probe.log)")
elif not c.get("train", {}).get("ok") or not c["train"].get("losses_finite"):
    reasons.append("train 12 step lỗi hoặc loss không hữu hạn")
h = est_hours(c.get("train"), "candidate", ds["n_fit_examples"])
if h is not None and h > 20:
    reasons.append(f"2 epoch train_fit cần ~{h} giờ/GPU (> 2 phiên Kaggle)")
if c.get("train", {}).get("ok") and not (c.get("merge", {}).get("ok") and c["merge"].get("n_missing") == 0):
    reasons.append("merge adapter vào checkpoint gốc lỗi")
if c.get("merge", {}).get("ok") and not (c.get("vllm", {}).get("ok") and c["vllm"].get("valid_json", 0) >= 0.9):
    reasons.append("vLLM không load được bản merge hoặc JSON hợp lệ < 90%")
SMOKE = {"experiment": CONFIG["experiment"], "verdict": "GO" if not reasons else "NO-GO", "reasons": reasons,
         "table": {n: [x, y] for n, x, y in rows_tbl}, "candidate": c, "baseline": b, "data": ds, "fla_installed": FLA_OK,
         "vllm_venv": VLLM_OK, "wall_min": round((time.time() - T0) / 60, 1)}
atomic_write_json(OUT / "SMOKE.json", SMOKE)
print("\nVERDICT:", SMOKE["verdict"], reasons)
for s in (c.get("vllm") or {}).get("samples", [])[:3]:
    print("  target:", s["target"], "\n  output:", s["output"])


In [ ]:
# Không lưu checkpoint / merge (GB) vào output: chỉ giữ report, log, SMOKE.json
for p in OUT.rglob("ckpt-*"):
    shutil.rmtree(p, ignore_errors=True)
for p in OUT.glob("*.jsonl"):
    p.unlink()
print(sorted(str(p.relative_to(OUT)) for p in OUT.rglob("*") if p.is_file())[:60])
